<a href="https://colab.research.google.com/github/AndikaPutra509/Prediksi-Saham/blob/main/Trading_and_Invest_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
📈 IDX STOCK SCANNER - QUADRUPLE ENGINE (FULL SYSTEM - AGGRESSIVE VERSION)
Fitur:
- Phase 1: Trading Scanner dengan Modal-Adaptive Filters (Data Warehouse)
- Phase 2: Walk-Forward Validation, Sensitivity Analysis
- Phase 2.3: Monte Carlo FIXED (realistis) - BUG FIXED
- Export: Google Sheets (LANGSUNG KE DRIVE, TANPA FILE)
- Panduan Portfolio: 3 rekomendasi terbaik dengan alokasi modal
- Risk per trade: 3.0% (AGGRESSIVE)
- Target return: 14-20% per tahun
- Market Regime dengan Confidence Score
- Error Handling di Data Warehouse (AMAN)

PERBAIKAN TERBARU:
- ✅ Look-ahead bias FIXED (semua indikator pakai data H-1)
- ✅ Kapasitas modal disesuaikan per engine
- ✅ Parameter adaptif terhadap regime & volatilitas
- ✅ Correlation check dengan stress testing
- ✅ Block size Monte Carlo optimal (data-driven)
- ✅ Error handling konsisten dengan logging
- ✅ OIL/GOLD NaN FIXED
- ✅ Dividend analyzer untuk Investasi Engine
- ✅ Portfolio guide aman terhadap None values
- ✅ HAPUS WALKFORWARD COLLECTOR (tidak ditampilkan)
- ✅ Tampilkan 5 rekomendasi terbaik (bukan 20)
- ✅ Portfolio tetap 3 saham terbaik

Author: Quant System
Date: 2026
"""

# =============================================================================
# 1. INSTALL DEPENDENCIES & IMPORTS
# =============================================================================

from google.colab import auth
from google.auth import default
import gspread
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
from datetime import datetime, timedelta
import time
import pickle
import os
import json
from tabulate import tabulate
from collections import defaultdict
import logging
import random
from typing import Optional, Dict, List, Tuple, Union, Any
import hashlib
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import statsmodels.api as sm

# Matikan logging yang tidak perlu
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

# Setup logging untuk error handling
logging.basicConfig(
    filename='scanner_errors.log',
    level=logging.ERROR,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Dependencies imported")

# =============================================================================
# 2. STOCK UNIVERSE (FULL) - TIDAK BERUBAH
# =============================================================================

STOCKBIT_UNIVERSE = [
    "AALI", "ABBA", "ABDA", "ABMM", "ACES", "ADES", "ADHI", "ADMF", "ADMG", "ADRO",
    "AGAR", "AGII", "AGRO", "AHAP", "AIMS", "AISA", "AKRA", "AKSI", "ALDO", "ALKA",
    "ALMI", "AMAG", "AMFG", "AMMN", "AMRT", "ANDI", "ANJT", "ANTM", "APEX", "APIC",
    "APLN", "ARCI", "ARGO", "ARII", "ARNA", "ARTA", "ARTO", "ASBI", "ASDM", "ASGR",
    "ASHA", "ASII", "ASLI", "ASMI", "ASPI", "ASRI", "ASRM", "ASSA", "ATLA", "AUTO",
    "AVIA", "AWAN", "AYLS", "BABP", "BACA", "BALI", "BANK", "BAPA", "BAPI", "BATA",
    "BAYU", "BBCA", "BBHI", "BBKP", "BBLD", "BBNI", "BBRI", "BBRM", "BBSS", "BBTN",
    "BBYB", "BCAP", "BCIC", "BDMN", "BEEF", "BEKS", "BELL", "BEST", "BFIN", "BGTG",
    "BHAT", "BIMA", "BINA", "BIPP", "BIPI", "BIRD", "BISI", "BJBR", "BJTM", "BKSL",
    "BLTA", "BLUE", "BMAS", "BMBL", "BMRI", "BMSR", "BMTR", "BNBA", "BNBR", "BNGA",
    "BNII", "BNLI", "BOBA", "BOLT", "BOSS", "BPFI", "BPII", "BPTR", "BRAM", "BREN",
    "BRIS", "BRMS", "BRNA", "BRPT", "BSDE", "BSIM", "BSSR", "BTEL", "BTON", "BTPN",
    "BTPS", "BUDI", "BULL", "BUMI", "BUVA", "BWPT", "BYAN", "CAMP", "CANI", "CARS",
    "CASA", "CASS", "CBDK", "CBMF", "CCSI", "CDAX", "CEKA", "CENT", "CFIN", "CITA",
    "CITY", "CKRA", "CLEO", "CLPI", "CMNP", "CMPP", "CMRY", "CNKO", "CNTX", "COAL",
    "COCO", "COWL", "CPIN", "CPRO", "CSAP", "CSIS", "CTBN", "CTRA", "CTTH", "CUAN",
    "DAAZ", "DART", "DASA", "DAYA", "DCII", "DEGA", "DEWA", "DFAM", "DGIK", "DIGI",
    "DILD", "DIVA", "DIVN", "DKFT", "DLTA", "DMAS", "DMND", "DNAR", "DNET", "DNLS",
    "DOID", "DOOH", "DPNS", "DPUM", "DSFI", "DSNG", "DSSA", "DUCK", "DUTI", "DVLA",
    "DYAN", "EASI", "EASY", "EBMT", "ECII", "EDGE", "EKAD", "ELBA", "ELSA", "ELTY",
    "EMBR", "EMDE", "EMTK", "ENRG", "ENVY", "ENZO", "EPAC", "EPMT", "ERAA", "ERTX",
    "ESSA", "ESTA", "ESTI", "ETWA", "EXCL", "FAST", "FASW", "FILM", "FISH", "FITT",
    "FKSF", "FLMC", "FMII", "FORE", "FORU", "FORZ", "FPNI", "FREN", "FUJI", "FUTR",
    "GAMA", "GDST", "GDYR", "GEMS", "GGRM", "GGRP", "GHON", "GIDS", "GJTL", "GLVA",
    "GMFI", "GMTD", "GOLD", "GOOD", "GOTO", "GPRA", "GRPH", "GSMF", "GTBO", "GTRA",
    "GTSI", "GULA", "GZCO", "HADE", "HDFA", "HDIT", "HEAL", "HERO", "HITS", "HKMU",
    "HMSP", "HOKI", "HOMI", "HOPE", "HOTL", "HRME", "HRTA", "HRUM", "HSBK", "HSMP",
    "HUMI", "IBFN", "IBOS", "IBST", "ICBP", "ICON", "IDPR", "IFII", "IFSH", "IGAR",
    "IIKP", "IKAI", "IKAN", "IMAS", "IMJS", "IMPC", "INAF", "INAI", "INCF", "INCI",
    "INCO", "INDF", "INDS", "INDX", "INDY", "INET", "INKP", "INPC", "INPP", "INPS",
    "INRU", "INTA", "INTD", "INTP", "IPCC", "IPCM", "IPOL", "IRRA", "ISAT", "ISEA",
    "ISSP", "ITIC", "ITMG", "JAST", "JAWA", "JAYA", "JECC", "JEMB", "JFAS", "JGLE",
    "JHON", "JIHD", "JKON", "JKSW", "JMAS", "JPFA", "JPII", "JPUR", "JRPT", "JSKY",
    "JSMR", "JSPT", "JTNB", "KAEF", "KAQI", "KARW", "KBLI", "KBLM", "KBRT", "KBRI",
    "KDSI", "KDTN", "KEEN", "KETR", "KICI", "KIJA", "KINO", "KIOS", "KJEN", "KKGI",
    "KLBF", "KMTR", "KOBX", "KOIN", "KOLI", "KONI", "KOTA", "KPAL", "KPIG", "KRAS",
    "KREN", "KRYA", "KSEL", "KUAS", "KUIC", "KUVO", "LAND", "LAPD", "LATA", "LBAK",
    "LCGP", "LCKM", "LEAD", "LIFE", "LINK", "LION", "LISA", "LMAS", "LMPI", "LMSH",
    "LPCK", "LPGI", "LPIN", "LPKR", "LPLI", "LPPF", "LPPS", "LSIP", "LSPI", "LTLS",
    "LUCY", "MABA", "MABH", "MAGP", "MAIN", "MAMI", "MAPA", "MAPB", "MAPI", "MARA",
    "MASA", "MAYA", "MBAP", "MBCA", "MBMA", "MBSS", "MBTO", "MCAS", "MCPI", "MCOR",
    "MDIA", "MDKA", "MDKI", "MEDC", "MEDS", "MEGA", "MERK", "META", "MFIN", "MFMI",
    "MGLV", "MGNA", "MGRO", "MIDI", "MIKA", "MINA", "MIRA", "MITI", "MITT", "MKNT",
    "MKPI", "MLBI", "MLIA", "MLPL", "MLPT", "MLSL", "MMIX", "MMLP", "MNCN", "MOLI",
    "MPOW", "MPPA", "MPRO", "MPTJ", "MRAT", "MSIE", "MSIN", "MSKY", "MTDL", "MTFN",
    "MTLA", "MTPS", "MTSM", "MUDA", "MUTU", "MYOH", "MYOR", "MYRX", "MYSX", "NAGA",
    "NASI", "NATO", "NAYZ", "NCKL", "NELY", "NETV", "NFCX", "NICL", "NIKL", "NISP",
    "NITY", "NIYM", "NOBU", "NPGF", "NRCA", "NSSS", "NTBK", "NUSA", "NUSI", "OASA",
    "OCTN", "OKAS", "OMED", "ONIX", "OPMS", "ORNA", "OTBK", "PADA", "PADI", "PAMG",
    "PANR", "PANS", "PANU", "PAPA", "PASA", "PASS", "PBRX", "PBID", "PBSA", "PCAR",
    "PDES", "PDGD", "PDIN", "PEGE", "PGAS", "PGLI", "PGUN", "PICO", "PIDRA", "PJAA",
    "PKPK", "PLAN", "PLAS", "PLIN", "PMJS", "PMMP", "PNBN", "PNBS", "PNIN", "PNLF",
    "PNSE", "POLI", "POLL", "POLU", "POLY", "POOL", "PORT", "POWR", "PPGL", "PPRE",
    "PPRO", "PPSI", "PRAS", "PRDA", "PRIM", "PRIN", "PRLD", "PROD", "PROT", "PRTS",
    "PSAB", "PSBA", "PSDN", "PSGO", "PSKT", "PSSI", "PTBA", "PTDU", "PTIS", "PTMP",
    "PTPP", "PTPW", "PTRO", "PTSN", "PTSP", "PUDP", "PURA", "PURE", "PWON", "PYFA",
    "RACE", "RADIO", "RAFI", "RAJA", "RAKD", "RALS", "RANC", "RATU", "RBMS", "RDTX",
    "REAL", "RELI", "RIGS", "RIMO", "RISE", "RMBA", "RMKE", "ROCK", "RODA", "ROKI",
    "ROTI", "RRMI", "RUIS", "RUMI", "SABA", "SAFE", "SAME", "SAPX", "SARA", "SATO",
    "SBAT", "SBBP", "SBGA", "SBMA", "SBMF", "SCBD", "SCCC", "SCCO", "SCMA", "SCNP",
    "SDPC", "SDRA", "SEAN", "SECR", "SEMA", "SFAN", "SGER", "SGRO", "SHID", "SHIP",
    "SIDO", "SILO", "SIMA", "SIMP", "SIPD", "SIPO", "SKBM", "SKLT", "SKRN", "SLIS",
    "SMAR", "SMDR", "SMGR", "SMIL", "SMMT", "SMSM", "SMRA", "SNLK", "SNMS", "SOFA",
    "SONA", "SOSS", "SOUL", "SPMA", "SPMI", "SPNA", "SPRE", "SPTO", "SQBI", "SQMI",
    "SRAJ", "SRIL", "SRSN", "SSIA", "SSMS", "SSTM", "STAR", "STTP", "SUGI", "SULI",
    "SUPR", "SURI", "SWAT", "SWID", "TALD", "TAMA", "TAMU", "TAPG", "TARA", "TASP",
    "TATA", "TAXI", "TBIG", "TBLA", "TCID", "TDPM", "TELE", "TEMB", "TEMPO", "TIFA",
    "TIGA", "TINS", "TIRA", "TIRT", "TITA", "TKGA", "TKIM", "TLKM", "TMAS", "TMPO",
    "TMSH", "TOBA", "TOOL", "TOPS", "TOSK", "TOTL", "TOTO", "TOWR", "TPIA", "TPMA",
    "TRAM", "TRGU", "TRIO", "TRIS", "TRJA", "TRON", "TRST", "TRUB", "TRUK", "TRUS",
    "TSPC", "TUGU", "TURI", "TUVN", "TYRE", "UANG", "UCID", "UDIJ", "UFNX", "UGRO",
    "UJSN", "ULTJ", "UNIC", "UNIQ", "UNIT", "UNSP", "USFI", "VALU", "VICO", "VICI",
    "VIDI", "VISI", "VIVA", "VKTR", "VOKS", "VRNA", "VTNY", "WAPO", "WEGE", "WEHA",
    "WICO", "WIFI", "WIIM", "WINS", "WMUU", "WMPP", "WOOD", "WOWS", "WRKR", "WSBP",
    "WSKT", "WTON", "YELO", "YULE", "ZBRA", "ZINC", "ZONE"
]

# =============================================================================
# 3. QUALITY STOCKS UNIVERSE (IDX30 + LQ45 + IDX80) - TIDAK BERUBAH
# =============================================================================

QUALITY_STOCKS = [
    'ADRO', 'AMRT', 'ANTM', 'ASII', 'BBCA', 'BBNI', 'BBRI', 'BMRI', 'BREN', 'BUMI',
    'CPIN', 'EMTK', 'GOTO', 'ICBP', 'INCO', 'INDF', 'INKP', 'ISAT', 'JPFA', 'JSMR',
    'KLBF', 'MDKA', 'MEDC', 'MIKA', 'MTEL', 'NCKL', 'PGAS', 'TLKM', 'TOWR', 'UNTR',
    'AKRA', 'ARTO', 'BBTN', 'BDKR', 'BIRD', 'BJBR', 'BJTM', 'BRIS', 'BRPT', 'BUKA',
    'EXCL', 'HRUM', 'INTP', 'ITMG', 'MPMX', 'PTBA', 'SIDO', 'SMGR', 'TPIA', 'UNVR',
    'WIKA', 'ACES', 'ADHI', 'AGRO', 'AALI', 'ARNA', 'ASGR', 'ASRI', 'AUTO', 'BAYU',
    'BEST', 'BFIN', 'BIPP', 'BISI', 'BKSL', 'BLTA', 'BMAS', 'BMSR', 'BNGA', 'BNII',
    'BOBA', 'BOLT', 'BOSS', 'BPFI', 'BRAM', 'BSDE', 'BSSR', 'BTON', 'BUDI', 'BULL',
    'BUVA', 'BWPT', 'BYAN', 'CAMP', 'CANI', 'CARS', 'CASA', 'CASS', 'CBDK', 'CBMF',
    'CCSI', 'CDAX', 'CEKA', 'CENT', 'CFIN', 'CITA', 'CITY', 'CKRA', 'CLEO', 'CLPI',
    'CMNP', 'CMPP', 'CMRY', 'CNKO', 'COAL', 'COCO', 'COWL', 'CPRO', 'CSAP', 'CSIS',
    'CTBN', 'CTRA', 'CTTH', 'CUAN', 'DART', 'DASA', 'DCII', 'DEGA', 'DEWA', 'DGIK',
    'DIGI', 'DILD', 'DIVA', 'DIVN', 'DLTA', 'DMAS', 'DMND', 'DNAR', 'DNET', 'DNLS',
    'DOID', 'DOOH', 'DPNS', 'DSFI', 'DSNG', 'DSSA', 'DUCK', 'DUTI', 'DVLA', 'DYAN',
    'EASI', 'EASY', 'EBMT', 'ECII', 'EDGE', 'EKAD', 'ELBA', 'ELSA', 'ELTY', 'EMBR',
    'EMDE', 'ENRG', 'ENVY', 'ENZO', 'EPAC', 'EPMT', 'ERAA', 'ESSA', 'ESTA', 'ESTI',
    'ETWA', 'FAST', 'FASW', 'FILM', 'FISH', 'FITT', 'FKSF', 'FLMC', 'FMII', 'FORE',
    'FORU', 'FORZ', 'FPNI', 'FREN', 'FUJI', 'FUTR', 'GAMA', 'GDST', 'GDYR', 'GEMS',
    'GGRM', 'GGRP', 'GHON', 'GIDS', 'GJTL', 'GLVA', 'GMFI', 'GMTD', 'GOLD', 'GOOD',
    'GPRA', 'GRPH', 'GSMF', 'GTBO', 'GTRA', 'GTSI', 'GULA', 'HADE', 'HDFA', 'HDIT',
    'HEAL', 'HERO', 'HITS', 'HKMU', 'HMSP', 'HOKI', 'HOMI', 'HOPE', 'HOTL', 'HRME',
    'HRTA', 'HSBK', 'HSMP', 'HUMI', 'IBFN', 'IBOS', 'IBST', 'ICON', 'IDPR', 'IFII',
    'IFSH', 'IGAR', 'IIKP', 'IKAI', 'IKAN', 'IMAS', 'IMJS', 'IMPC', 'INAF', 'INAI',
    'INCF', 'INCI', 'INDS', 'INDX', 'INDY', 'INET', 'INKP', 'INPC', 'INPP', 'INPS',
    'INRU', 'INTA', 'INTD', 'INTP', 'IPCC', 'IPCM', 'IPOL', 'IRRA', 'ISEA', 'ISSP',
    'ITIC', 'JAST', 'JAWA', 'JAYA', 'JECC', 'JEMB', 'JFAS', 'JGLE', 'JHON', 'JIHD',
    'JKON', 'JKSW', 'JMAS', 'JPII', 'JPUR', 'JRPT', 'JSKY', 'JSPT', 'JTNB', 'KAEF',
    'KAQI', 'KARW', 'KBLI', 'KBLM', 'KBRT', 'KBRI', 'KDSI', 'KDTN', 'KEEN', 'KETR',
    'KICI', 'KIJA', 'KINO', 'KIOS', 'KJEN', 'KKGI', 'KMTR', 'KOBX', 'KOIN', 'KOLI',
    'KONI', 'KOTA', 'KPAL', 'KPIG', 'KRAS', 'KREN', 'KRYA', 'KSEL', 'KUAS', 'KUIC',
    'KUVO', 'LAND', 'LAPD', 'LATA', 'LBAK', 'LCGP', 'LCKM', 'LEAD', 'LIFE', 'LINK',
    'LION', 'LISA', 'LMAS', 'LMPI', 'LMSH', 'LPCK', 'LPGI', 'LPIN', 'LPKR', 'LPLI',
    'LPPF', 'LPPS', 'LSIP', 'LSPI', 'LTLS', 'LUCY', 'MABA', 'MABH', 'MAGP', 'MAIN',
    'MAMI', 'MAPA', 'MAPB', 'MAPI', 'MARA', 'MASA', 'MAYA', 'MBAP', 'MBCA', 'MBMA',
    'MBSS', 'MBTO', 'MCAS', 'MCPI', 'MCOR', 'MDIA', 'MDKA', 'MDKI', 'MEDC', 'MEDS',
    'MEGA', 'MERK', 'META', 'MFIN', 'MFMI', 'MGLV', 'MGNA', 'MGRO', 'MIDI', 'MIKA',
    'MINA', 'MIRA', 'MITI', 'MITT', 'MKNT', 'MKPI', 'MLBI', 'MLIA', 'MLPL', 'MLPT',
    'MLSL', 'MMIX', 'MMLP', 'MNCN', 'MOLI', 'MPOW', 'MPPA', 'MPRO', 'MPTJ', 'MRAT',
    'MSIE', 'MSIN', 'MSKY', 'MTDL', 'MTFN', 'MTLA', 'MTPS', 'MTSM', 'MUDA', 'MUTU',
    'MYOH', 'MYOR', 'MYRX', 'MYSX', 'NAGA', 'NASI', 'NATO', 'NAYZ', 'NCKL', 'NELY',
    'NETV', 'NFCX', 'NICL', 'NIKL', 'NISP', 'NITY', 'NIYM', 'NOBU', 'NPGF', 'NRCA',
    'NSSS', 'NTBK', 'NUSA', 'NUSI', 'OASA', 'OCTN', 'OKAS', 'OMED', 'ONIX', 'OPMS',
    'ORNA', 'OTBK', 'PADA', 'PADI', 'PAMG', 'PANR', 'PANS', 'PANU', 'PAPA', 'PASA',
    'PASS', 'PBRX', 'PBID', 'PBSA', 'PCAR', 'PDES', 'PDGD', 'PDIN', 'PEGE', 'PGAS',
    'PGLI', 'PGUN', 'PICO', 'PIDRA', 'PJAA', 'PKPK', 'PLAN', 'PLAS', 'PLIN', 'PMJS',
    'PMMP', 'PNBN', 'PNBS', 'PNIN', 'PNLF', 'PNSE', 'POLI', 'POLL', 'POLU', 'POLY',
    'POOL', 'PORT', 'POWR', 'PPGL', 'PPRE', 'PPRO', 'PPSI', 'PRAS', 'PRDA', 'PRIM',
    'PRIN', 'PRLD', 'PROD', 'PROT', 'PRTS', 'PSAB', 'PSBA', 'PSDN', 'PSGO', 'PSKT',
    'PSSI', 'PTDU', 'PTIS', 'PTMP', 'PTPP', 'PTPW', 'PTRO', 'PTSN', 'PTSP', 'PUDP',
    'PURA', 'PURE', 'PWON', 'PYFA', 'RACE', 'RADIO', 'RAFI', 'RAJA', 'RAKD', 'RALS',
    'RANC', 'RATU', 'RBMS', 'RDTX', 'REAL', 'RELI', 'RIGS', 'RIMO', 'RISE', 'RMBA',
    'RMKE', 'ROCK', 'RODA', 'ROKI', 'ROTI', 'RRMI', 'RUIS', 'RUMI', 'SABA', 'SAFE',
    'SAME', 'SAPX', 'SARA', 'SATO', 'SBAT', 'SBBP', 'SBGA', 'SBMA', 'SBMF', 'SCBD',
    'SCCC', 'SCCO', 'SCMA', 'SCNP', 'SDPC', 'SDRA', 'SEAN', 'SECR', 'SEMA', 'SFAN',
    'SGER', 'SGRO', 'SHID', 'SHIP', 'SIDO', 'SILO', 'SIMA', 'SIMP', 'SIPD', 'SIPO',
    'SKBM', 'SKLT', 'SKRN', 'SLIS', 'SMAR', 'SMDR', 'SMGR', 'SMIL', 'SMMT', 'SMSM',
    'SMRA', 'SNLK', 'SNMS', 'SOFA', 'SONA', 'SOSS', 'SOUL', 'SPMA', 'SPMI', 'SPNA',
    'SPRE', 'SPTO', 'SQBI', 'SQMI', 'SRAJ', 'SRIL', 'SRSN', 'SSIA', 'SSMS', 'SSTM',
    'STAR', 'STTP', 'SUGI', 'SULI', 'SUPR', 'SURI', 'SWAT', 'SWID', 'TALD', 'TAMA',
    'TAMU', 'TAPG', 'TARA', 'TASP', 'TATA', 'TAXI', 'TBIG', 'TBLA', 'TCID', 'TDPM',
    'TELE', 'TEMB', 'TEMPO', 'TIFA', 'TIGA', 'TINS', 'TIRA', 'TIRT', 'TITA', 'TKGA',
    'TKIM', 'TLKM', 'TMAS', 'TMPO', 'TMSH', 'TOBA', 'TOOL', 'TOPS', 'TOSK', 'TOTL',
    'TOTO', 'TPMA', 'TRAM', 'TRGU', 'TRIO', 'TRIS', 'TRJA', 'TRON', 'TRST', 'TRUB',
    'TRUK', 'TRUS', 'TSPC', 'TUGU', 'TURI', 'TUVN', 'TYRE', 'UANG', 'UCID', 'UDIJ',
    'UFNX', 'UGRO', 'UJSN', 'ULTJ', 'UNIC', 'UNIQ', 'UNIT', 'UNSP', 'USFI', 'VALU',
    'VICO', 'VICI', 'VIDI', 'VISI', 'VIVA', 'VKTR', 'VOKS', 'VRNA', 'VTNY', 'WAPO',
    'WEGE', 'WEHA', 'WICO', 'WIFI', 'WIIM', 'WINS', 'WMUU', 'WMPP', 'WOOD', 'WOWS',
    'WRKR', 'WSBP', 'WSKT', 'WTON', 'YELO', 'YULE', 'ZBRA', 'ZINC', 'ZONE'
]
QUALITY_STOCKS = list(dict.fromkeys(QUALITY_STOCKS))

# =============================================================================
# 4. SECTOR MAPPING - TIDAK BERUBAH
# =============================================================================

ENERGY_SECTOR = ['ADRO', 'PTBA', 'ITMG', 'MEDC', 'PGAS', 'ENRG', 'BUMI', 'DOID']
MINING_GOLD = ['ANTM', 'MDKA', 'BRMS']
EXPORT_SECTOR = ['ANTM', 'INCO', 'TINS', 'HRUM', 'CPIN', 'JPFA', 'MAIN']

SECTOR_MAPPING = {
    'ENERGY': ENERGY_SECTOR + ['BYAN', 'INDY', 'HRUM', 'ARTI'],
    'MINING': MINING_GOLD + ['INCO', 'TINS', 'CITA', 'DKFT'],
    'FINANCE': ['BBCA', 'BBRI', 'BMRI', 'BBNI', 'PNBN', 'BJBR', 'BJTM', 'NISP', 'BDMN', 'BNLI', 'BNGA', 'BNII', 'BSIM'],
    'PROPERTY': ['PWON', 'BSDE', 'LPKR', 'CTRA', 'SMRA', 'ASRI', 'DILD', 'MDLN', 'ELTY', 'BIPP', 'BKSL', 'MTLA', 'MAPI'],
    'CONSUMER': ['UNVR', 'ICBP', 'INDF', 'GGRM', 'HMSP', 'KLBF', 'MYOR', 'SIDO', 'ULTJ', 'DLTA', 'MLBI', 'TCID', 'ROTI', 'SKBM'],
    'INFRASTRUCTURE': ['TLKM', 'JSMR', 'TOWR', 'TBIG', 'WIKA', 'PTPP', 'WSKT', 'ADHI', 'TOTL', 'ACST'],
    'INDUSTRIAL': ['ASII', 'GJTL', 'AUTO', 'BRAM', 'INDS', 'BOLT', 'IMP', 'PRAS', 'PBRX'],
    'TRADE': ['MAPI', 'ACES', 'RALS', 'LPPF', 'ERAA', 'MAPB', 'SONA', 'CSAP', 'MIDI', 'MFMI'],
    'AGRICULTURE': ['AALI', 'LSIP', 'SGRO', 'BWPT', 'SMAR', 'DSNG', 'JAWA'],
    'HEALTHCARE': ['MIKA', 'SILO', 'KLBF', 'KAEF', 'INAF', 'DVLA', 'TSPC', 'MERK', 'SCPI']
}

def get_sector(symbol: str) -> str:
    """Get sector for a symbol"""
    for sector, stocks in SECTOR_MAPPING.items():
        if symbol in stocks:
            return sector
    return 'OTHER'

# =============================================================================
# 5. GLOBAL INDICES CONFIGURATION - TIDAK BERUBAH
# =============================================================================

GLOBAL_INDICES = {
    "IHSG": {
        "ticker": "^JKSE",
        "nama": "Indeks Harga Saham Gabungan",
        "satuan": "poin",
        "keterangan": "Indeks utama BEI"
    },
    "DOWJONES": {
        "ticker": "^DJI",
        "nama": "Dow Jones Industrial Average",
        "satuan": "poin",
        "keterangan": "Indeks saham AS"
    },
    "USDIDR": {
        "ticker": "IDR=X",
        "nama": "USD/IDR",
        "satuan": "Rp/USD",
        "keterangan": "Nilai tukar Rupiah terhadap Dolar"
    },
    "OIL": {
        "ticker": "CL=F",
        "nama": "Crude Oil WTI",
        "satuan": "USD/barel",
        "keterangan": "Harga minyak dunia"
    },
    "GOLD": {
        "ticker": "GC=F",
        "nama": "Gold Futures",
        "satuan": "USD/ons",
        "keterangan": "Harga emas dunia"
    }
}

# =============================================================================
# 6. GLOBAL INDICES FETCHER (LENGKAP) - VERSI FIX UNTUK OIL/GOLD
# =============================================================================

class GlobalIndicesFetcher:
    """Fetch global indices data for market context - DENGAN FIX UNTUK OIL/GOLD"""

    def __init__(self):
        self.data = {}
        self.momentum = {}
        self.status = {}
        self.prices = {}
        self.detail = {}

    def fetch_all(self) -> None:
        print("\n" + "="*80)
        print("📡 FETCHING GLOBAL INDICES - DETAIL TRANSPARAN")
        print("="*80)

        end_date = datetime.now()
        start_date = end_date - timedelta(days=365*2)

        for name, config in GLOBAL_INDICES.items():
            ticker = config["ticker"]
            print(f"\n📊 {name}: {config['nama']}")
            print(f"   Ticker: {ticker}")
            print(f"   Satuan: {config['satuan']}")
            print(f"   Keterangan: {config['keterangan']}")

            try:
                df = yf.download(
                    ticker,
                    start=start_date.strftime("%Y-%m-%d"),
                    end=end_date.strftime("%Y-%m-%d"),
                    interval="1d",
                    auto_adjust=True,
                    progress=False,
                    timeout=10
                )

                time.sleep(0.5)

                if df.empty or len(df) < 200:
                    self.status[name] = "UNAVAILABLE"
                    self.momentum[name] = 0.0
                    self.prices[name] = 0.0
                    print(f"   ⚠️  Status: TIDAK TERSEDIA (data tidak cukup)")
                else:
                    # Normalize columns
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = df.columns.get_level_values(0)

                    # FIX UNTUK OIL & GOLD: handle NaN
                    if name in ["OIL", "GOLD"]:
                        # Coba ambil dari kolom yang tersedia
                        close_col = None
                        for col in ['Close', 'Adj Close']:
                            if col in df.columns and not df[col].isna().all():
                                close_col = col
                                break

                        if close_col is None:
                            # Jika tidak ada, ambil kolom numerik pertama
                            for col in df.columns:
                                if pd.api.types.is_numeric_dtype(df[col]) and not df[col].isna().all():
                                    close_col = col
                                    break

                        if close_col:
                            close = df[close_col].values
                            # Filter NaN
                            close = close[~np.isnan(close)]
                        else:
                            close = np.array([])
                    else:
                        close = df['Close'].values

                    if len(close) == 0:
                        self.status[name] = "UNAVAILABLE"
                        self.momentum[name] = 0.0
                        self.prices[name] = 0.0
                        print(f"   ⚠️  Status: TIDAK TERSEDIA (data tidak valid)")
                        continue

                    current_price = float(close[-1])
                    price_1w_ago = float(close[-6]) if len(close) >= 6 else current_price
                    price_1m_ago = float(close[-22]) if len(close) >= 22 else current_price
                    price_3m_ago = float(close[-66]) if len(close) >= 66 else current_price
                    price_1y_ago = float(close[-252]) if len(close) >= 252 else current_price

                    momentum_1w = (current_price / price_1w_ago - 1) * 100 if price_1w_ago > 0 else 0
                    momentum_1m = (current_price / price_1m_ago - 1) * 100 if price_1m_ago > 0 else 0
                    momentum_3m = (current_price / price_3m_ago - 1) * 100 if price_3m_ago > 0 else 0
                    momentum_1y = (current_price / price_1y_ago - 1) * 100 if price_1y_ago > 0 else 0

                    if name == "IHSG" and len(close) >= 200:
                        ma50 = np.mean(close[-50:])
                        ma200 = np.mean(close[-200:])
                        self.data['IHSG_MA50'] = ma50
                        self.data['IHSG_MA200'] = ma200
                        self.data['IHSG_Close'] = current_price

                        if current_price > ma50 > ma200:
                            trend = "BULLISH (di atas MA50 & MA200)"
                        elif current_price < ma50 < ma200:
                            trend = "BEARISH (di bawah MA50 & MA200)"
                        elif current_price > ma50:
                            trend = "NETRAL (di atas MA50, di bawah MA200)"
                        else:
                            trend = "NETRAL (di bawah MA50, di atas MA200)"

                        returns = pd.Series(close).pct_change().dropna() * 100
                        volatility = returns.tail(20).std() * np.sqrt(252) if len(returns) > 20 else 20.0
                    else:
                        trend = "TIDAK TERSEDIA"
                        volatility = 20.0

                    self.detail[name] = {
                        'current_price': current_price,
                        'price_1w_ago': price_1w_ago,
                        'price_1m_ago': price_1m_ago,
                        'price_3m_ago': price_3m_ago,
                        'price_1y_ago': price_1y_ago,
                        'momentum_1w': round(momentum_1w, 2),
                        'momentum_1m': round(momentum_1m, 2),
                        'momentum_3m': round(momentum_3m, 2),
                        'momentum_1y': round(momentum_1y, 2),
                        'trend': trend,
                        'volatility': round(volatility, 2),
                        'data_length': len(df)
                    }

                    self.data[name] = df
                    self.momentum[name] = round(momentum_1m, 2)
                    self.prices[name] = round(current_price, 2)
                    self.status[name] = "OK"

                    print(f"   ✅ Status: TERSEDIA")
                    print(f"   Harga Saat Ini: {self.get_price_str(name)}")
                    print(f"   Momentum 1w: {momentum_1w:+.2f}%")
                    print(f"   Momentum 1m: {momentum_1m:+.2f}%")
                    print(f"   Momentum 3m: {momentum_3m:+.2f}%")
                    print(f"   Momentum 1y: {momentum_1y:+.2f}%")
                    if name == "IHSG":
                        print(f"   MA50: {ma50:,.2f}")
                        print(f"   MA200: {ma200:,.2f}")
                        print(f"   Trend: {trend}")
                        print(f"   Volatility (annualized): {volatility:.1f}%")

            except Exception as e:
                self.status[name] = "ERROR"
                self.momentum[name] = 0.0
                self.prices[name] = 0.0
                logger.error(f"Error fetching {name}: {str(e)}")
                print(f"   ❌ Status: ERROR - {str(e)[:50]}")
                time.sleep(0.5)

        print("\n" + "="*80)
        print("✅ SEMUA INDEKS GLOBAL SELESAI DI-FETCH")
        print("="*80)

    def print_detailed_report(self) -> None:
        """Tampilkan laporan detail indeks global"""
        if not self.detail:
            print("\n📊 Tidak ada data indeks untuk ditampilkan")
            return

        print("\n" + "="*100)
        print("📊 LAPORAN DETAIL INDEKS GLOBAL")
        print("="*100)

        data = []
        for name, detail in self.detail.items():
            data.append([
                name,
                self.get_price_str(name),
                f"{detail['momentum_1w']:+.2f}%",
                f"{detail['momentum_1m']:+.2f}%",
                f"{detail['momentum_3m']:+.2f}%",
                f"{detail['momentum_1y']:+.2f}%",
                detail.get('trend', 'N/A'),
                detail.get('volatility', 'N/A'),
                self.status[name]
            ])

        headers = ["Indeks", "Harga", "1W", "1M", "3M", "1Y", "Trend", "Vol%", "Status"]
        print(tabulate(data, headers=headers, tablefmt='grid'))

    def get_price_str(self, name: str) -> str:
        """Dapatkan string harga dengan format sesuai jenis indeks"""
        price = self.prices.get(name, 0)
        if price == 0:
            return "N/A"

        if name in ["IHSG", "DOWJONES"]:
            return f"{price:,.2f}"
        elif name == "USDIDR":
            return f"Rp {price:,.0f}"
        elif name == "OIL":
            return f"US$ {price:.2f}"
        elif name == "GOLD":
            return f"US$ {price:.2f}"
        return f"{price:.2f}"

    def get_trend(self, name: str) -> str:
        """Dapatkan trend berdasarkan momentum"""
        mom = self.momentum.get(name, 0)
        if mom > 0.5:
            return "🟢 BULLISH"
        elif mom < -0.5:
            return "🔴 BEARISH"
        else:
            return "🟡 NETRAL"

    def get_momentum(self, name: str) -> float:
        """Dapatkan momentum 1 bulan untuk indeks tertentu"""
        return self.momentum.get(name, 0.0)

    def get_price(self, name: str) -> float:
        """Dapatkan harga terkini untuk indeks tertentu"""
        return self.prices.get(name, 0.0)

    def is_ihsg_bullish(self) -> bool:
        """Cek apakah IHSG dalam kondisi bullish (di atas MA200)"""
        if 'IHSG_Close' in self.data and 'IHSG_MA200' in self.data:
            return self.data['IHSG_Close'] > self.data['IHSG_MA200']
        return True

# =============================================================================
# 6A. MARKET REGIME DETECTOR (DENGAN CONFIDENCE SCORE) - DIPERBAIKI
# =============================================================================

class MarketRegimeDetector:
    """
    Deteksi regime pasar: BULL, BEAR, SIDEWAYS, HIGH_VOL
    Dilengkapi dengan CONFIDENCE SCORE (0-100%)
    MENGGUNAKAN DATA H-1 UNTUK SEMUA KALKULASI
    """

    def __init__(self, lookback_days: int = 252):
        self.lookback_days = lookback_days
        self.current_regime = "UNKNOWN"
        self.confidence = 0.0
        self.regime_params = {}
        self.regime_history = []

    def detect_regime(self, ihsg_data: pd.DataFrame) -> Tuple[str, float]:
        if ihsg_data is None or len(ihsg_data) < 200:
            return "UNKNOWN", 0.0

        df = ihsg_data.shift(1).copy()
        close = df['Close'].dropna()

        if len(close) < 200:
            return "UNKNOWN", 0.0

        returns = close.pct_change().dropna() * 100

        mom_20d = (close.iloc[-1] / close.iloc[-21] - 1) * 100 if len(close) > 21 else 0
        mom_60d = (close.iloc[-1] / close.iloc[-61] - 1) * 100 if len(close) > 61 else 0

        vol_20d = returns.tail(20).std() * np.sqrt(252)
        vol_60d = returns.tail(60).std() * np.sqrt(252)
        hist_vol = returns.tail(252).std() * np.sqrt(252) if len(returns) > 252 else 20

        ma50 = close.rolling(50).mean()
        ma200 = close.rolling(200).mean()

        price_vs_ma50 = (close.iloc[-1] / ma50.iloc[-1] - 1) * 100 if not pd.isna(ma50.iloc[-1]) else 0
        price_vs_ma200 = (close.iloc[-1] / ma200.iloc[-1] - 1) * 100 if not pd.isna(ma200.iloc[-1]) else 0

        high = df['High']
        low = df['Low']

        tr1 = high - low
        tr2 = (high - close.shift()).abs()
        tr3 = (low - close.shift()).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.rolling(14).mean()

        up_move = high - high.shift()
        down_move = low.shift() - low
        plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
        plus_di = 100 * (pd.Series(plus_dm).rolling(14).mean() / atr)

        trend_strength = plus_di.iloc[-1] if not pd.isna(plus_di.iloc[-1]) else 25

        if vol_20d > hist_vol * 1.5:
            regime = "HIGH_VOL"
            vol_ratio = vol_20d / hist_vol
            confidence = min(100, vol_ratio * 50)

        elif mom_20d > 2 and mom_60d > 5 and price_vs_ma200 > 0:
            regime = "BULL"
            mom_strength = min(100, (abs(mom_20d) + abs(mom_60d)) * 5)
            ma_strength = min(100, price_vs_ma200 * 10)
            trend_conf = min(100, trend_strength * 2)
            confidence = (mom_strength * 0.4 + ma_strength * 0.3 + trend_conf * 0.3)

        elif mom_20d < -2 and mom_60d < -5 and price_vs_ma200 < 0:
            regime = "BEAR"
            mom_strength = min(100, (abs(mom_20d) + abs(mom_60d)) * 5)
            ma_strength = min(100, abs(price_vs_ma200) * 10)
            trend_conf = min(100, trend_strength * 2)
            confidence = (mom_strength * 0.4 + ma_strength * 0.3 + trend_conf * 0.3)

        elif abs(mom_20d) < 3 and abs(mom_60d) < 5:
            regime = "SIDEWAYS"
            mom_weakness = 100 - min(100, (abs(mom_20d) + abs(mom_60d)) * 10)
            vol_weakness = 100 - min(100, (vol_20d / hist_vol) * 50)
            trend_weakness = 100 - min(100, trend_strength * 2)
            confidence = (mom_weakness * 0.4 + vol_weakness * 0.3 + trend_weakness * 0.3)

        else:
            regime = "NEUTRAL"
            confidence = 30.0

        confidence = round(min(confidence, 100.0), 1)

        self.current_regime = regime
        self.confidence = confidence
        self.regime_history.append({
            'date': datetime.now(),
            'regime': regime,
            'confidence': confidence,
            'mom_20d': mom_20d,
            'mom_60d': mom_60d,
            'vol_20d': vol_20d,
            'price_vs_ma200': price_vs_ma200,
            'trend_strength': trend_strength
        })

        if len(self.regime_history) > 30:
            self.regime_history = self.regime_history[-30:]

        return regime, confidence

    def get_regime_parameters(self, base_risk: float = 3.0) -> Dict:
        confidence_factor = self.confidence / 100.0

        if self.current_regime == "BULL":
            base_multiplier = 1.17
            risk_multiplier = 1.0 + (base_multiplier - 1.0) * confidence_factor

            return {
                'risk_multiplier': round(risk_multiplier, 2),
                'entry_tolerance_multiplier': 0.8,
                'min_rr_multiplier': 1.2,
                'max_positions': 5,
                'description': f'BULL MARKET - Aggressive (Confidence: {self.confidence}%)'
            }
        elif self.current_regime == "BEAR":
            base_multiplier = 0.67
            risk_multiplier = 1.0 - (1.0 - base_multiplier) * confidence_factor

            return {
                'risk_multiplier': round(risk_multiplier, 2),
                'entry_tolerance_multiplier': 1.5,
                'min_rr_multiplier': 0.8,
                'max_positions': 3,
                'description': f'BEAR MARKET - Defensive (Confidence: {self.confidence}%)'
            }
        elif self.current_regime == "HIGH_VOL":
            base_multiplier = 0.5
            risk_multiplier = 1.0 - (1.0 - base_multiplier) * confidence_factor

            return {
                'risk_multiplier': round(risk_multiplier, 2),
                'entry_tolerance_multiplier': 2.0,
                'min_rr_multiplier': 1.5,
                'max_positions': 2,
                'description': f'HIGH VOLATILITY - Very Defensive (Confidence: {self.confidence}%)'
            }
        elif self.current_regime == "SIDEWAYS":
            base_multiplier = 0.83
            risk_multiplier = 1.0 - (1.0 - base_multiplier) * confidence_factor

            return {
                'risk_multiplier': round(risk_multiplier, 2),
                'entry_tolerance_multiplier': 1.2,
                'min_rr_multiplier': 1.0,
                'max_positions': 4,
                'description': f'SIDEWAYS - Neutral (Confidence: {self.confidence}%)'
            }
        else:
            return {
                'risk_multiplier': 1.0,
                'entry_tolerance_multiplier': 1.0,
                'min_rr_multiplier': 1.0,
                'max_positions': 4,
                'description': f'NEUTRAL - Standard (Confidence: {self.confidence}%)'
            }

    def print_regime_report(self) -> None:
        print("\n" + "="*70)
        print("📊 MARKET REGIME DETECTION (WITH CONFIDENCE)")
        print("="*70)

        params = self.get_regime_parameters()

        confidence_bar = '█' * int(self.confidence / 5) + '░' * (20 - int(self.confidence / 5))

        print(f"Current Regime: {self.current_regime}")
        print(f"Confidence: {self.confidence}% [{confidence_bar}]")
        print(f"Description: {params['description']}")
        print(f"\nAdjusted Parameters:")
        print(f"  - Risk Multiplier: {params['risk_multiplier']:.2f}x")
        print(f"  - Entry Tolerance: {params['entry_tolerance_multiplier']:.1f}x")
        print(f"  - Min R/R Multiplier: {params['min_rr_multiplier']:.1f}x")
        print(f"  - Max Positions: {params['max_positions']}")

        if len(self.regime_history) > 1:
            print(f"\nRegime History (last {len(self.regime_history)}):")
            print(f"{'Date':12} | {'Regime':10} | {'Conf':6}")
            print("-" * 35)
            for h in self.regime_history[-5:]:
                print(f"{h['date'].strftime('%Y-%m-%d')} | {h['regime']:10} | {h['confidence']:5.1f}%")

# =============================================================================
# 7. DATA WAREHOUSE (DENGAN ERROR HANDLING AMAN) - DIPERBAIKI (FOLDER DIVIDEN TERPISAH)
# =============================================================================

class DataWarehouse:
    """Menyimpan data historis LENGKAP untuk semua saham - DENGAN LOGGING"""

    # DATA WAREHOUSE UTAMA TIDAK BERUBAH - AMAN

    def __init__(self, warehouse_dir: str = 'data_warehouse', min_days: int = 400):
        self.warehouse_dir = warehouse_dir
        self.min_days = min_days
        os.makedirs(warehouse_dir, exist_ok=True)

        # FOLDER TERPISAH UNTUK DIVIDEN - TIDAK MENGGANGGU DATA UTAMA
        self.dividend_dir = f'{warehouse_dir}/dividends'
        os.makedirs(self.dividend_dir, exist_ok=True)

        self.stats = {
            'total_symbols': 0,
            'downloaded': 0,
            'failed': 0,
            'cached': 0,
            'filtered_min_days': 0,
            'corrupt_files': 0
        }

    def download_complete_history(
        self,
        symbols: List[str],
        start_date: str = '2018-01-01',
        end_date: str = '2026-12-31',
        force_refresh: bool = False
    ) -> Dict[str, pd.DataFrame]:
        print("\n" + "="*80)
        print("🗄️  DATA WAREHOUSE - DOWNLOAD HISTORIS LENGKAP")
        print("="*80)
        print(f"Periode: {start_date} hingga {end_date}")
        print(f"Total saham: {len(symbols)}")
        print(f"Minimal hari: {self.min_days} (saham dengan data kurang akan difilter)")

        results = {}
        self.stats['total_symbols'] = len(symbols)

        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)

        for i, symbol in enumerate(symbols):
            cache_file = f"{self.warehouse_dir}/{symbol}_full.parquet"

            if os.path.exists(cache_file) and not force_refresh:
                try:
                    df = pd.read_parquet(cache_file)

                    if len(df) >= self.min_days and df.index[0] <= start_dt and df.index[-1] >= end_dt:
                        results[symbol] = df
                        self.stats['cached'] += 1
                    else:
                        self.stats['filtered_min_days'] += 1

                    if (i + 1) % 50 == 0:
                        print(f"   Progress: {i+1}/{len(symbols)} - {len(results)} dimuat (cache)")
                    continue
                except Exception as e:
                    logger.error(f"Corrupt cache file for {symbol}: {str(e)}")
                    self.stats['corrupt_files'] += 1

            try:
                ticker = f"{symbol}.JK"
                print(f"   Downloading {symbol} ({i+1}/{len(symbols)})...", end=" ")

                df = yf.download(
                    ticker,
                    start=start_date,
                    end=end_date,
                    interval="1d",
                    auto_adjust=True,
                    progress=False,
                    timeout=30
                )

                time.sleep(0.5)

                if df.empty:
                    print("❌ GAGAL (data kosong)")
                    self.stats['failed'] += 1
                    logger.warning(f"Empty data for {symbol}")
                    continue

                df = normalize_columns(df)

                if len(df) < self.min_days:
                    print(f"❌ GAGAL (hanya {len(df)} hari, minimal {self.min_days})")
                    self.stats['filtered_min_days'] += 1
                    continue

                df.to_parquet(cache_file)
                results[symbol] = df
                self.stats['downloaded'] += 1
                print(f"✅ {len(df)} hari")

            except Exception as e:
                print(f"❌ ERROR: {str(e)[:50]}")
                self.stats['failed'] += 1
                logger.error(f"Error downloading {symbol}: {str(e)}")
                time.sleep(1)

        print("\n" + "="*80)
        print("📊 RINGKASAN DATA WAREHOUSE")
        print("="*80)
        print(f"Total saham: {self.stats['total_symbols']}")
        print(f"Berhasil dimuat: {len(results)}")
        print(f"  - Dari cache: {self.stats['cached']}")
        print(f"  - Download baru: {self.stats['downloaded']}")
        print(f"  - Difilter (< {self.min_days} hari): {self.stats['filtered_min_days']}")
        print(f"  - File corrupt: {self.stats['corrupt_files']}")
        print(f"Gagal: {self.stats['failed']}")
        print("="*80)

        return results

    def get_all_valid_symbols(self) -> List[str]:
        symbols = []
        for f in os.listdir(self.warehouse_dir):
            if f.endswith('_full.parquet'):
                symbol = f.replace('_full.parquet', '')
                try:
                    df = pd.read_parquet(os.path.join(self.warehouse_dir, f))
                    if len(df) >= self.min_days:
                        symbols.append(symbol)
                except Exception as e:
                    logger.error(f"Corrupt file in get_all_valid_symbols: {f} - {str(e)}")
                    continue
        return symbols

    def get_data(self, symbol: str) -> Optional[pd.DataFrame]:
        cache_file = f"{self.warehouse_dir}/{symbol}_full.parquet"

        if not os.path.exists(cache_file):
            return None

        try:
            df = pd.read_parquet(cache_file)
            if len(df) >= self.min_days:
                return df
            return None
        except Exception as e:
            logger.error(f"Error reading {symbol}: {str(e)}")
            return None

    def get_all_data(self, max_symbols: int = None) -> Dict[str, pd.DataFrame]:
        results = {}
        symbols = self.get_all_valid_symbols()

        if max_symbols:
            symbols = symbols[:max_symbols]

        for symbol in symbols:
            df = self.get_data(symbol)
            if df is not None:
                results[symbol] = df

        return results

    def print_warehouse_stats(self) -> None:
        symbols = self.get_all_valid_symbols()

        print("\n" + "="*80)
        print("🗄️  DATA WAREHOUSE STATISTICS")
        print("="*80)
        print(f"Total saham valid (≥{self.min_days} hari): {len(symbols)}")

        if symbols:
            sample = symbols[0]
            df = pd.read_parquet(f"{self.warehouse_dir}/{sample}_full.parquet")
            print(f"Rentang tanggal: {df.index[0].date()} hingga {df.index[-1].date()}")
            print(f"Rata-rata hari per saham: {len(df)} hari")

        print("="*80)

    # ==================== DIVIDEN METHODS (FOLDER TERPISAH) ====================

    def download_dividend_history(self, symbols: List[str], years_back: int = 10) -> Dict[str, pd.DataFrame]:
        """
        Download historical dividend data untuk semua saham
        DISIMPAN DI FOLDER TERPISAH - TIDAK MENGGANGGU DATA UTAMA
        """
        print("\n" + "="*80)
        print("💰 DOWNLOADING DIVIDEND HISTORY (FOLDER TERPISAH)")
        print("="*80)

        results = {}
        end_date = datetime.now()
        start_date = end_date - timedelta(days=365*years_back)

        for i, symbol in enumerate(symbols):
            cache_file = f"{self.dividend_dir}/{symbol}_dividends.parquet"

            if os.path.exists(cache_file):
                try:
                    df = pd.read_parquet(cache_file)
                    results[symbol] = df
                    if (i+1) % 50 == 0:
                        print(f"   Progress: {i+1}/{len(symbols)} - {len(results)} from cache")
                    continue
                except:
                    pass

            try:
                ticker = yf.Ticker(f"{symbol}.JK")
                dividends = ticker.dividends

                if dividends is not None and len(dividends) > 0:
                    df = pd.DataFrame(dividends)
                    df.columns = ['Dividend']
                    df.index.name = 'Date'

                    df = df[df.index >= pd.Timestamp(start_date)]

                    if len(df) > 0:
                        df.to_parquet(cache_file)
                        results[symbol] = df
                        print(f"   ✅ {symbol}: {len(df)} dividend records")
                    else:
                        print(f"   ⚠️ {symbol}: no dividends in period")
                else:
                    print(f"   ⚠️ {symbol}: no dividend data")

                time.sleep(0.2)

            except Exception as e:
                print(f"   ❌ {symbol}: {str(e)[:50]}")
                logger.error(f"Error downloading dividend for {symbol}: {str(e)}")

        print("\n" + "="*80)
        print(f"✅ Dividend download complete: {len(results)} symbols with dividends")
        print("="*80)

        return results

    def get_dividends(self, symbol: str) -> Optional[pd.DataFrame]:
        """Ambil data dividen dari folder terpisah"""
        cache_file = f"{self.dividend_dir}/{symbol}_dividends.parquet"
        if os.path.exists(cache_file):
            try:
                return pd.read_parquet(cache_file)
            except:
                return None
        return None

# =============================================================================
# 8. UTILITY FUNCTIONS - TIDAK BERUBAH
# =============================================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

def calculate_spread_pct(df: pd.DataFrame) -> float:
    try:
        spread = ((df['High'] - df['Low']) / df['Close']).tail(10).mean() * 100
        return float(spread)
    except Exception:
        return 999.0

def calculate_return(series: pd.Series, period: int = 5) -> float:
    if len(series) < period + 1:
        return 0.0
    return (series.iloc[-1] / series.iloc[-period-1] - 1) * 100

# =============================================================================
# 9. BACKTEST METRICS (UNTUK WALKFORWARD - TAPI TIDAK DIGUNAKAN DI UI)
# =============================================================================

class BacktestMetrics:
    """
    Menyimpan dan menghitung metrics backtest untuk sinyal live
    TIDAK DITAMPILKAN DI UI - HANYA UNTUK INTERNAL
    """

    def __init__(self, min_trades: int = 20):
        self.min_trades = min_trades
        self.trades = []

    def add_trade(self, return_pct: float) -> None:
        self.trades.append(return_pct)
        if len(self.trades) > 100:
            self.trades = self.trades[-100:]

    def calculate_metrics(self) -> Dict:
        if len(self.trades) < self.min_trades:
            return {
                'has_data': False,
                'total_trades': len(self.trades),
                'trades_needed': self.min_trades - len(self.trades),
                'display': f"⏳ ({len(self.trades)}/{self.min_trades})"
            }

        trades_array = np.array(self.trades)
        winning_trades = trades_array[trades_array > 0]

        win_rate = len(winning_trades) / len(trades_array) * 100

        return {
            'has_data': True,
            'total_trades': len(self.trades),
            'win_rate': round(win_rate, 1),
            'display': f"{round(win_rate,1)}% ({len(self.trades)} trades)"
        }

# =============================================================================
# 10. RISK MANAGER (AGGRESSIVE - 3% BASE RISK) - DIPERBAIKI
# =============================================================================

class RiskManager:
    def __init__(
        self,
        modal: float,
        risk_per_trade_pct: float = 3.0,
        max_risk_portfolio_pct: float = 15.0,
        max_lot_per_position: int = 10,
        engine_type: str = 'swing'
    ):
        self.modal = modal
        self.base_risk_per_trade_pct = risk_per_trade_pct
        self.risk_per_trade_pct = risk_per_trade_pct
        self.max_risk_portfolio_pct = max_risk_portfolio_pct
        self.max_lot_per_position = max_lot_per_position
        self.engine_type = engine_type

        self._validate_modal_for_engine()

        self.risk_per_trade_rp = modal * (risk_per_trade_pct / 100)
        self.max_risk_portfolio_rp = modal * (max_risk_portfolio_pct / 100)

        self.current_positions = []
        self.current_risk_rp = 0.0

        self.trade_history = []
        self.win_rate = 50.0
        self.avg_win = 0
        self.avg_loss = 0
        self.regime_params = {}

        self.backtest_metrics = {}

    def _validate_modal_for_engine(self) -> None:
        min_modal_map = {
            'swing': 40000,
            'intraday_liquid': 10000,
            'intraday_gorengan': 10000,
            'investasi': 100000
        }

        max_modal_map = {
            'swing': 5000000,
            'intraday_liquid': 500000,
            'intraday_gorengan': 500000,
            'investasi': 1000000
        }

        min_modal = min_modal_map.get(self.engine_type, 10000)
        max_modal = max_modal_map.get(self.engine_type, 5000000)

        if self.modal < min_modal:
            raise ValueError(f"Modal minimal untuk {self.engine_type} adalah Rp {min_modal:,}")
        if self.modal > max_modal:
            raise ValueError(f"Modal maksimal untuk {self.engine_type} adalah Rp {max_modal:,}")

    def calculate_atr_in_rupiah(self, df: pd.DataFrame, period: int = 14) -> float:
        try:
            high = df['High'].shift(1)
            low = df['Low'].shift(1)
            close = df['Close'].shift(1)

            tr1 = high - low
            tr2 = (high - close).abs()
            tr3 = (low - close).abs()

            tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
            atr = tr.rolling(window=period, min_periods=period).mean()

            valid_atr = atr.dropna()
            if len(valid_atr) == 0:
                return df['Close'].iloc[-1] * 0.02

            return float(valid_atr.iloc[-1])

        except Exception as e:
            logger.error(f"Error calculating ATR: {str(e)}")
            return df['Close'].iloc[-1] * 0.02

    def calculate_kelly_lot(self, close: float, atr: float, symbol: str = None) -> tuple:
        if atr <= 0 or close <= 0:
            return None, None, None

        self._update_trade_stats()

        if symbol and symbol in self.backtest_metrics:
            bt = self.backtest_metrics[symbol].calculate_metrics()
            if bt.get('has_data', False):
                win_rate = 0.7 * bt['win_rate'] + 0.3 * self.win_rate
                avg_win = bt.get('avg_return', self.avg_win)
                avg_loss = abs(bt.get('avg_return', self.avg_loss)) if bt.get('avg_return', 0) < 0 else self.avg_loss
            else:
                win_rate = self.win_rate
                avg_win = self.avg_win
                avg_loss = self.avg_loss
        else:
            win_rate = self.win_rate
            avg_win = self.avg_win
            avg_loss = self.avg_loss

        if avg_loss > 0 and win_rate > 0:
            b = avg_win / avg_loss if avg_win > 0 else 1
            p = win_rate / 100
            q = 1 - p

            if b > 0:
                kelly = (p * b - q) / b
                kelly = max(0, min(kelly, 0.25))
            else:
                kelly = 0.01
        else:
            kelly = 0.01

        adjusted_risk_pct = self.risk_per_trade_pct * (1 + kelly * 4)
        adjusted_risk_pct = min(adjusted_risk_pct, 6.0)

        risk_per_trade_rp = self.modal * (adjusted_risk_pct / 100)
        risk_per_lot = atr * 100

        raw_lot = risk_per_trade_rp / risk_per_lot
        lot = int(raw_lot)

        lot = min(lot, self.max_lot_per_position)
        max_lot_by_modal = int(self.modal / (close * 100))
        lot = min(lot, max_lot_by_modal)

        if lot < 1:
            return None, None, None

        cost = lot * 100 * close
        risk_amount = lot * risk_per_lot

        return lot, cost, risk_amount

    def calculate_lot(
        self,
        close: float,
        atr: float,
        symbol: str = None,
        use_kelly: bool = True
    ):
        if use_kelly and (len(self.trade_history) >= 10 or (symbol and symbol in self.backtest_metrics)):
            return self.calculate_kelly_lot(close, atr, symbol)

        if atr <= 0 or close <= 0:
            return None, None, None

        risk_per_lot = atr * 100
        raw_lot = self.risk_per_trade_rp / risk_per_lot
        lot = int(raw_lot)

        lot = min(lot, self.max_lot_per_position)
        max_lot_by_modal = int(self.modal / (close * 100))
        lot = min(lot, max_lot_by_modal)

        if lot < 1:
            return None, None, None

        cost = lot * 100 * close
        risk_amount = lot * risk_per_lot

        return lot, cost, risk_amount

    def _update_trade_stats(self):
        if len(self.trade_history) < 5:
            return

        profits = [t['return_pct'] for t in self.trade_history if t['return_pct'] > 0]
        losses = [abs(t['return_pct']) for t in self.trade_history if t['return_pct'] <= 0]

        self.win_rate = (len(profits) / len(self.trade_history)) * 100 if self.trade_history else 50

        self.avg_win = np.mean(profits) if profits else 0
        self.avg_loss = np.mean(losses) if losses else 0

    def add_trade_result(self, trade_result: Dict):
        self.trade_history.append(trade_result)
        if len(self.trade_history) > 100:
            self.trade_history = self.trade_history[-100:]

    def add_backtest_metrics(self, symbol: str, metrics: BacktestMetrics) -> None:
        self.backtest_metrics[symbol] = metrics

    def adjust_for_regime(self, regime_params: Dict):
        self.regime_params = regime_params
        multiplier = regime_params.get('risk_multiplier', 1.0)
        self.risk_per_trade_pct = self.base_risk_per_trade_pct * multiplier
        self.risk_per_trade_rp = self.modal * (self.risk_per_trade_pct / 100)

    def can_add_position(self, risk_amount: float) -> Tuple[bool, str]:
        if self.current_risk_rp + risk_amount > self.max_risk_portfolio_rp:
            return False, f"Portfolio risk limit: Rp {self.max_risk_portfolio_rp:,.0f}"

        if self.regime_params and 'max_positions' in self.regime_params:
            if len(self.current_positions) >= self.regime_params['max_positions']:
                return False, f"Max positions ({self.regime_params['max_positions']}) reached"

        return True, "OK"

    def add_position(self, symbol: str, lot: int, entry_price: float, risk_amount: float) -> None:
        position = {
            'symbol': symbol,
            'lot': lot,
            'entry_price': entry_price,
            'risk_amount': risk_amount,
            'entry_date': datetime.now()
        }
        self.current_positions.append(position)
        self.current_risk_rp += risk_amount

    def remove_position(self, symbol: str) -> None:
        for i, pos in enumerate(self.current_positions):
            if pos['symbol'] == symbol:
                self.current_risk_rp -= pos['risk_amount']
                self.current_positions.pop(i)
                break

    def get_portfolio_risk_pct(self) -> float:
        return (self.current_risk_rp / self.modal) * 100 if self.modal > 0 else 0

    def reset(self) -> None:
        self.current_positions = []
        self.current_risk_rp = 0.0

# =============================================================================
# 11. PORTFOLIO RISK CALCULATOR (DENGAN CORRELATION CHECK & STRESS TESTING) - DIPERBAIKI
# =============================================================================

class PortfolioRiskCalculator:
    def __init__(self, lookback_days: int = 60):
        self.lookback_days = lookback_days
        self.correlation_cache = {}
        self.stress_scenarios = {
            'covid_2020': {'shock': -0.30, 'correlation_multiplier': 1.5},
            'taper_tantrum_2013': {'shock': -0.15, 'correlation_multiplier': 1.3},
            'normal': {'shock': 0, 'correlation_multiplier': 1.0}
        }

    def calculate_correlation_matrix(
        self,
        symbols: List[str],
        price_data: Dict[str, pd.DataFrame],
        use_exp_weight: bool = True
    ) -> pd.DataFrame:
        returns_dict = {}

        for symbol in symbols:
            if symbol in price_data:
                df = price_data[symbol]
                if len(df) >= self.lookback_days:
                    returns = df['Close'].pct_change().shift(1).tail(self.lookback_days)
                    returns_dict[symbol] = returns

        if not returns_dict:
            return pd.DataFrame()

        returns_df = pd.DataFrame(returns_dict)

        if use_exp_weight and len(returns_df) > 30:
            decay_factor = 0.94
            weights = decay_factor ** np.arange(len(returns_df))[::-1]
            weights = weights / weights.sum()

            mean_returns = returns_df.mean()
            demeaned = returns_df - mean_returns
            cov_matrix = np.dot((demeaned.T * weights), demeaned) / (1 - decay_factor)

            std_dev = np.sqrt(np.diag(cov_matrix))
            corr_matrix = cov_matrix / np.outer(std_dev, std_dev)
            corr_df = pd.DataFrame(corr_matrix, index=returns_df.columns, columns=returns_df.columns)
        else:
            corr_df = returns_df.corr()

        return corr_df.fillna(0)

    def get_correlation(self, symbol1: str, symbol2: str, price_data: Dict[str, pd.DataFrame]) -> float:
        cache_key = f"{symbol1}_{symbol2}"
        if cache_key in self.correlation_cache:
            return self.correlation_cache[cache_key]

        if symbol1 not in price_data or symbol2 not in price_data:
            return 0.0

        df1 = price_data[symbol1]
        df2 = price_data[symbol2]

        if len(df1) < self.lookback_days or len(df2) < self.lookback_days:
            return 0.0

        returns1 = df1['Close'].pct_change().shift(1).tail(self.lookback_days)
        returns2 = df2['Close'].pct_change().shift(1).tail(self.lookback_days)

        corr = returns1.corr(returns2)
        if pd.isna(corr):
            corr = 0.0

        self.correlation_cache[cache_key] = corr
        self.correlation_cache[f"{symbol2}_{symbol1}"] = corr

        return corr

    def calculate_stressed_correlation(self, corr_matrix: pd.DataFrame, scenario: str = 'covid_2020') -> pd.DataFrame:
        if scenario not in self.stress_scenarios:
            scenario = 'normal'

        params = self.stress_scenarios[scenario]

        stressed_corr = corr_matrix.copy()

        np.fill_diagonal(stressed_corr.values, 1)

        mask = ~np.eye(*stressed_corr.shape, dtype=bool)
        stressed_corr.values[mask] = np.minimum(
            stressed_corr.values[mask] * params['correlation_multiplier'],
            0.99
        )

        return stressed_corr

    def check_correlation_risk(
        self,
        new_symbol: str,
        existing_positions: List[Dict],
        price_data: Dict[str, pd.DataFrame],
        max_correlation: float = 0.7,
        use_stressed: bool = True
    ) -> Tuple[bool, str]:
        if not existing_positions:
            return True, "OK"

        for pos in existing_positions:
            symbol = pos['symbol']
            corr = self.get_correlation(new_symbol, symbol, price_data)

            if use_stressed:
                stressed_corr = min(corr * 1.3, 0.99)

            check_corr = stressed_corr if use_stressed else corr

            if check_corr > max_correlation:
                return False, f"Korelasi {corr:.2f} (stress: {check_corr:.2f}) dengan {symbol} > {max_correlation}"

        return True, "OK"

    def calculate_portfolio_var(
        self,
        positions: Dict[str, Dict],
        corr_matrix: pd.DataFrame,
        confidence: float = 0.95,
        use_stressed: bool = False
    ) -> Tuple[float, float, float]:
        if not positions or corr_matrix.empty:
            return 0.0, 0.0, 0.0

        symbols = list(positions.keys())
        weights = []
        total_risk = sum([p['risk_amount'] for p in positions.values()])

        for symbol in symbols:
            if symbol in positions and symbol in corr_matrix.index:
                weight = positions[symbol]['risk_amount'] / total_risk if total_risk > 0 else 0
                weights.append(weight)

        if len(weights) < 2:
            z_score = {0.95: 1.645, 0.99: 2.326}[confidence]
            var_amount = total_risk * z_score
            return var_amount, total_risk, 1.0

        try:
            sub_corr = corr_matrix.loc[symbols, symbols].values

            if use_stressed:
                mask = ~np.eye(len(symbols), dtype=bool)
                sub_corr[mask] = np.minimum(sub_corr[mask] * 1.3, 0.99)

        except Exception:
            return total_risk * 1.645, total_risk, 1.0

        weights_array = np.array(weights)
        portfolio_variance = np.dot(np.dot(weights_array, sub_corr), weights_array)
        portfolio_std = np.sqrt(max(portfolio_variance, 0.001))

        z_score = {0.95: 1.645, 0.99: 2.326}[confidence]
        var_amount = total_risk * portfolio_std * z_score

        return var_amount, total_risk, portfolio_std

# =============================================================================
# 12. REALISTIC FEE CONFIG - TIDAK BERUBAH
# =============================================================================

class RealisticFeeConfig:
    BASE_BROKER_FEE_BUY = 0.0015
    BASE_BROKER_FEE_SELL = 0.0025
    EXCHANGE_FEE = 0.0001
    VAT_RATE = 0.11

    BROKER_FEE_BUY = BASE_BROKER_FEE_BUY * (1 + VAT_RATE)
    BROKER_FEE_SELL = BASE_BROKER_FEE_SELL * (1 + VAT_RATE)

    SLIPPAGE_MODEL = {
        'high': {'buy': 0.0005, 'sell': 0.0010},
        'medium': {'buy': 0.0010, 'sell': 0.0020},
        'low': {'buy': 0.0020, 'sell': 0.0030}
    }

    MIN_FEE_PER_TRANSACTION = 10000

    def __init__(self, liquidity: str = 'medium'):
        self.liquidity = liquidity
        self.slippage = self.SLIPPAGE_MODEL.get(liquidity, self.SLIPPAGE_MODEL['medium'])

    def calculate_buy_cost(self, price: float, lot: int) -> float:
        amount = price * 100 * lot
        broker_fee = amount * self.BROKER_FEE_BUY
        exchange_fee = amount * self.EXCHANGE_FEE
        slippage = amount * self.slippage['buy']
        total_fee = broker_fee + exchange_fee + slippage
        return max(total_fee, self.MIN_FEE_PER_TRANSACTION)

    def calculate_sell_cost(self, price: float, lot: int) -> float:
        amount = price * 100 * lot
        broker_fee = amount * self.BROKER_FEE_SELL
        exchange_fee = amount * self.EXCHANGE_FEE
        slippage = amount * self.slippage['sell']
        total_fee = broker_fee + exchange_fee + slippage
        return max(total_fee, self.MIN_FEE_PER_TRANSACTION)

    def calculate_round_trip(
        self,
        entry_price: float,
        exit_price: float,
        lot: int
    ) -> Tuple[float, float, float]:
        buy_cost = self.calculate_buy_cost(entry_price, lot)
        sell_cost = self.calculate_sell_cost(exit_price, lot)
        total_fee = buy_cost + sell_cost

        buy_amount = entry_price * 100 * lot
        sell_amount = exit_price * 100 * lot

        gross_profit = sell_amount - buy_amount
        net_profit = gross_profit - total_fee
        net_return_pct = (net_profit / buy_amount) * 100 if buy_amount > 0 else 0

        return total_fee, net_profit, net_return_pct

# =============================================================================
# 13. ENTRY DELAY SIMULATOR - DIPERBAIKI (HANYA UNTUK INFORMASI)
# =============================================================================

class EntryDelaySimulator:
    """
    Simulasi entry delay - HANYA UNTUK INFORMASI, BUKAN FILTER
    """

    def __init__(self, max_delay: int = 2):
        self.max_delay = max_delay

    def simulate_entry(
        self,
        df: pd.DataFrame,
        signal_idx: int,
        signal_price: float
    ) -> Optional[Dict]:
        start_idx = signal_idx + 1
        end_idx = min(start_idx + self.max_delay, len(df) - 1)

        entry_candidates = []

        for idx in range(start_idx, end_idx + 1):
            row = df.iloc[idx]
            open_price = float(row['Open'])
            low_price = float(row['Low'])

            if open_price <= signal_price * 1.01:
                entry_candidates.append({
                    'idx': idx,
                    'price': open_price,
                    'delay': idx - signal_idx,
                    'type': 'open'
                })

            if low_price <= signal_price:
                entry_price = min(signal_price, open_price)
                entry_candidates.append({
                    'idx': idx,
                    'price': entry_price,
                    'delay': idx - signal_idx,
                    'type': 'limit'
                })

        if entry_candidates:
            best = min(entry_candidates, key=lambda x: x['price'])
            best['slippage_pct'] = ((best['price'] - signal_price) / signal_price) * 100
            return best

        return None

# =============================================================================
# 14. DIVIDEN ANALYZER (BARU)
# =============================================================================

class DividendAnalyzer:
    """
    Analisis kualitas dividen untuk Investasi Engine
    TANPA DEFAULT VALUES - semua berdasarkan data real
    """

    def __init__(self):
        self.weights = {
            'yield': 0.30,
            'consistency': 0.25,
            'growth': 0.25,
            'payout': 0.20
        }

        # Threshold untuk kategorisasi
        self.thresholds = {
            'aristocrat': 85,
            'quality': 70,
            'decent': 50,
            'speculative': 30
        }

    def analyze_dividend_yield(self, yield_pct: float) -> Dict:
        """Analisis dividend yield"""
        if yield_pct <= 0:
            return {'score': 0, 'grade': 'NO_YIELD', 'action': 'Tidak bagi dividen'}

        if yield_pct > 8:
            return {
                'score': 100,
                'grade': 'EXCELLENT',
                'warning': 'Yield sangat tinggi, perlu cek sustainabilitas',
                'action': 'Menarik tapi waspada'
            }
        elif yield_pct > 6:
            return {
                'score': 90,
                'grade': 'VERY_GOOD',
                'action': 'Sangat menarik untuk income'
            }
        elif yield_pct > 4:
            return {
                'score': 80,
                'grade': 'GOOD',
                'action': 'Di atas rata-rata IHSG'
            }
        elif yield_pct > 2:
            return {
                'score': 60,
                'grade': 'FAIR',
                'action': 'Rata-rata pasar'
            }
        else:
            return {
                'score': 40,
                'grade': 'LOW',
                'action': 'Dividen kecil'
            }

    def analyze_consistency(self, dividend_series: pd.Series) -> Dict:
        """Analisis konsistensi pembayaran dividen"""
        if len(dividend_series) < 2:
            return {'score': 0, 'consistent_years': 0, 'grade': 'INSUFFICIENT_DATA'}

        yearly_div = dividend_series.resample('Y').sum()
        years_with_div = (yearly_div > 0).sum()
        total_years = len(yearly_div)

        consistency_ratio = years_with_div / total_years if total_years > 0 else 0

        streak = 0
        max_streak = 0
        for has_div in (yearly_div > 0):
            if has_div:
                streak += 1
                max_streak = max(max_streak, streak)
            else:
                streak = 0

        if consistency_ratio > 0.9 and max_streak > 10:
            return {
                'score': 100,
                'grade': 'BLUE_CHIP',
                'consistent_years': max_streak,
                'action': 'Dividend king, sangat reliable'
            }
        elif consistency_ratio > 0.8 and max_streak > 5:
            return {
                'score': 80,
                'grade': 'CONSISTENT',
                'consistent_years': max_streak,
                'action': 'Reliable dividend payer'
            }
        elif consistency_ratio > 0.6:
            return {
                'score': 60,
                'grade': 'MODERATE',
                'consistent_years': max_streak,
                'action': 'Cukup konsisten'
            }
        else:
            return {
                'score': 30,
                'grade': 'IRREGULAR',
                'consistent_years': max_streak,
                'action': 'Dividen tidak rutin'
            }

    def analyze_growth(self, dividend_series: pd.Series) -> Dict:
        """Analisis pertumbuhan dividen"""
        if len(dividend_series) < 3:
            return {'score': 0, 'cagr': 0, 'grade': 'INSUFFICIENT_DATA'}

        yearly_div = dividend_series.resample('Y').sum()
        yearly_div = yearly_div[yearly_div > 0]

        if len(yearly_div) < 3:
            return {'score': 0, 'cagr': 0, 'grade': 'INSUFFICIENT_DATA'}

        first_year = yearly_div.iloc[0]
        last_year = yearly_div.iloc[-1]
        years = len(yearly_div) - 1

        if first_year > 0 and years > 0:
            cagr = (last_year / first_year) ** (1/years) - 1
            cagr_pct = cagr * 100
        else:
            cagr_pct = 0

        is_growing = all(yearly_div.iloc[i] <= yearly_div.iloc[i+1]
                         for i in range(len(yearly_div)-1))

        if cagr_pct > 10 and is_growing:
            return {
                'score': 100,
                'cagr': round(cagr_pct, 2),
                'grade': 'EXCELLENT_GROWTH',
                'action': 'Dividen tumbuh cepat'
            }
        elif cagr_pct > 5:
            return {
                'score': 80,
                'cagr': round(cagr_pct, 2),
                'grade': 'GOOD_GROWTH',
                'action': 'Dividen tumbuh stabil'
            }
        elif cagr_pct > 0:
            return {
                'score': 60,
                'cagr': round(cagr_pct, 2),
                'grade': 'SLOW_GROWTH',
                'action': 'Dividen tumbuh lambat'
            }
        else:
            return {
                'score': 30,
                'cagr': round(cagr_pct, 2),
                'grade': 'DECLINING',
                'action': 'Dividen menurun'
            }

    def estimate_payout_ratio(self, dividend: float, price: float) -> Dict:
        """Estimasi payout ratio (asumsi PER 15x)"""
        if dividend <= 0 or price <= 0:
            return {'score': 0, 'payout': 0, 'grade': 'NO_DATA'}

        estimated_eps = price / 15
        payout = (dividend / estimated_eps) * 100 if estimated_eps > 0 else 0

        if payout > 100:
            return {
                'score': 20,
                'payout': round(payout, 1),
                'grade': 'VERY_HIGH',
                'warning': 'Dividen > laba, tidak sustain'
            }
        elif payout > 80:
            return {
                'score': 50,
                'payout': round(payout, 1),
                'grade': 'HIGH',
                'warning': 'Payout ratio tinggi'
            }
        elif payout > 50:
            return {
                'score': 80,
                'payout': round(payout, 1),
                'grade': 'HEALTHY',
                'note': 'Payout ratio sehat'
            }
        elif payout > 20:
            return {
                'score': 100,
                'payout': round(payout, 1),
                'grade': 'VERY_HEALTHY',
                'note': 'Ruang untuk naik'
            }
        else:
            return {
                'score': 60,
                'payout': round(payout, 1),
                'grade': 'LOW',
                'note': 'Payout rendah, mungkin growth stock'
            }

    def analyze(self, symbol: str, dividend_df: pd.DataFrame, current_price: float) -> Dict:
        """
        Analisis komprehensif kualitas dividen
        TANPA DEFAULT VALUES - berdasarkan data real
        """
        if dividend_df is None or len(dividend_df) == 0:
            return {
                'has_dividend': False,
                'total_score': 0,
                'category': 'NON_DIVIDEND',
                'suitability': ['Growth Investor', 'Aggressive Growth'],
                'display': 'NO_DIVIDEND'
            }

        dividend_series = dividend_df['Dividend']

        annual_dividend = dividend_series.last('365D').sum()
        if current_price > 0 and annual_dividend > 0:
            yield_pct = (annual_dividend / current_price) * 100
        else:
            yield_pct = 0

        yield_result = self.analyze_dividend_yield(yield_pct)
        consistency_result = self.analyze_consistency(dividend_series)
        growth_result = self.analyze_growth(dividend_series)
        payout_result = self.estimate_payout_ratio(annual_dividend, current_price)

        total_score = (
            yield_result['score'] * self.weights['yield'] +
            consistency_result['score'] * self.weights['consistency'] +
            growth_result['score'] * self.weights['growth'] +
            payout_result['score'] * self.weights['payout']
        )

        # Kategorisasi
        if total_score >= self.thresholds['aristocrat']:
            category = 'DIVIDEND_ARISTOCRAT'
            suitability = ['Income Investor', 'Retirement Fund', 'Conservative']
            display = '🏆 ARISTOCRAT'
        elif total_score >= self.thresholds['quality']:
            category = 'QUALITY_DIVIDEND'
            suitability = ['Income Investor', 'Balanced Portfolio']
            display = '⭐ QUALITY'
        elif total_score >= self.thresholds['decent']:
            category = 'DECENT_DIVIDEND'
            suitability = ['Growth & Income', 'Moderate Risk']
            display = '✅ DECENT'
        elif total_score >= self.thresholds['speculative']:
            category = 'SPECULATIVE_DIVIDEND'
            suitability = ['Yield-Seeking', 'High Risk']
            display = '⚠️ SPECULATIVE'
        else:
            category = 'NON_DIVIDEND'
            suitability = ['Growth Investor', 'Aggressive Growth']
            display = '📈 GROWTH'

        return {
            'has_dividend': True,
            'symbol': symbol,
            'total_score': round(total_score, 1),
            'category': category,
            'display': display,
            'suitability': suitability,
            'metrics': {
                'dividend_yield': round(yield_pct, 2),
                'annual_dividend': round(annual_dividend, 2),
                'consistency_years': consistency_result.get('consistent_years', 0),
                'dividend_growth': growth_result.get('cagr', 0),
                'estimated_payout': payout_result.get('payout', 0),
            }
        }

# =============================================================================
# 15. BASE STRATEGY ENGINE (DENGAN REGIME PARAMETERS) - DIPERBAIKI
# =============================================================================

class BaseStrategyEngine:
    def __init__(self, config: Any, global_fetcher: GlobalIndicesFetcher, engine_type: str):
        self.config = config
        self.global_fetcher = global_fetcher
        self.engine_type = engine_type
        self.risk_manager = None
        self.regime_detector = None
        self.current_regime_params = {}

        self.backtest_metrics = {}

    def set_risk_manager(self, risk_manager: RiskManager) -> None:
        self.risk_manager = risk_manager

    def set_regime_detector(self, regime_detector: MarketRegimeDetector) -> None:
        self.regime_detector = regime_detector
        if regime_detector:
            self.current_regime_params = regime_detector.get_regime_parameters()
            if self.risk_manager:
                self.risk_manager.adjust_for_regime(self.current_regime_params)

    def apply_regime_parameters(self, base_params: Dict) -> Dict:
        if not self.current_regime_params:
            return base_params

        multiplier = self.current_regime_params.get('entry_tolerance_multiplier', 1.0)
        rr_multiplier = self.current_regime_params.get('min_rr_multiplier', 1.0)

        adjusted = base_params.copy()

        if 'entry_tolerance' in adjusted:
            adjusted['entry_tolerance'] = adjusted['entry_tolerance'] * multiplier

        if 'min_rr' in adjusted:
            adjusted['min_rr'] = adjusted['min_rr'] * rr_multiplier

        return adjusted

    def get_signal(self, symbol: str, df: pd.DataFrame):
        raise NotImplementedError

# =============================================================================
# 16. MODAL ADAPTER (DENGAN KAPASITAS PER ENGINE) - DIPERBAIKI
# =============================================================================

class ModalAdapter:
    def __init__(self, modal: float, engine_type: str):
        self.modal = modal
        self.engine_type = engine_type

        if modal < 50000:
            self.kategori = "ULTRA_MIKRO"
        elif modal < 100000:
            self.kategori = "MIKRO"
        elif modal < 500000:
            self.kategori = "RETAIL"
        elif modal < 2000000:
            self.kategori = "MENENGAH"
        else:
            self.kategori = "INSTITUSI"

        self.max_harga_beli = modal / 100

    def get_filter_harga(self) -> Tuple[float, float]:
        if self.engine_type == 'swing':
            min_price = 50
            max_price = min(5000, self.max_harga_beli)
        elif self.engine_type == 'intraday_liquid':
            min_price = 10
            max_price = min(1000, self.max_harga_beli)
        elif self.engine_type == 'intraday_gorengan':
            min_price = 5
            max_price = min(500, self.max_harga_beli)
        elif self.engine_type == 'investasi':
            min_price = 100
            max_price = min(3000, self.max_harga_beli)
        else:
            min_price = 50
            max_price = min(1000, self.max_harga_beli)

        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            min_price = max(min_price, 5)
        elif self.kategori == "RETAIL":
            min_price = max(min_price, 10)
        elif self.kategori == "MENENGAH":
            min_price = max(min_price, 50)
        else:
            min_price = max(min_price, 100)

        return min_price, max_price

    def get_filter_volume(self) -> Tuple[float, float]:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 5000, 100000
        elif self.kategori == "RETAIL":
            return 15000, 200000
        elif self.kategori == "MENENGAH":
            return 30000, 500000
        else:
            return 50000, 1000000

    def get_filter_spread(self) -> float:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 8.0
        elif self.kategori == "RETAIL":
            return 6.0
        elif self.kategori == "MENENGAH":
            return 4.0
        else:
            return 3.0

    def get_filter_min_history(self) -> int:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 30
        elif self.kategori == "RETAIL":
            return 60
        elif self.kategori == "MENENGAH":
            return 100
        else:
            return 150

    def get_quality_filter(self) -> str:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO", "RETAIL"]:
            return "ALL"
        elif self.kategori == "MENENGAH":
            return "LQ45"
        else:
            return "QUALITY"

    def get_entry_tolerance(self) -> float:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 15.0
        elif self.kategori == "RETAIL":
            return 10.0
        elif self.kategori == "MENENGAH":
            return 7.0
        else:
            return 3.0

    def get_trend_tolerance(self) -> float:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 0.85
        elif self.kategori == "RETAIL":
            return 0.90
        elif self.kategori == "MENENGAH":
            return 0.95
        else:
            return 0.98

    def print_info(self) -> None:
        min_price, max_price = self.get_filter_harga()
        min_vol, max_vol = self.get_filter_volume()

        print(f"\n📊 ADAPTASI FILTER UNTUK MODAL: {self.kategori} - ENGINE: {self.engine_type}")
        print(f"   Kapasitas modal: Rp {self.modal:,}")
        print(f"   Max harga per saham: Rp {self.max_harga_beli:,.0f}")
        print(f"   Filter harga: Rp {min_price:,.0f} - {max_price:,.0f}")
        print(f"   Filter volume: {min_vol:,} - {max_vol:,} lembar")
        print(f"   Max spread: {self.get_filter_spread()}%")
        print(f"   Minimal history: {self.get_filter_min_history()} hari")
        print(f"   Quality filter: {self.get_quality_filter()}")
        print(f"   Entry tolerance: {self.get_entry_tolerance()}% dari MA50")
        print(f"   Trend tolerance: {self.get_trend_tolerance()*100:.0f}% dari MA200")

# =============================================================================
# 17. INVESTASI ENGINE (DENGAN DIVIDEN ANALYZER) - DIPERBAIKI
# =============================================================================

class InvestasiEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher, engine_type='investasi')
        self.quality_stocks = QUALITY_STOCKS
        self.modal_adapter = ModalAdapter(config.MODAL, self.engine_type)
        self.warehouse = None
        self.dividend_analyzer = DividendAnalyzer()

        self.weights = {
            'teknikal': 0.40,
            'dividen': 0.35,
            'risk': 0.15,
            'backtest': 0.10
        }

    def set_warehouse(self, warehouse):
        self.warehouse = warehouse

    def calculate_target_prices(self, current_price: float, atr: float, df: pd.DataFrame) -> Dict:
        close = df['Close'].shift(1)

        ma50 = close.rolling(50).mean()
        ma200 = close.rolling(200).mean()

        current_ma50 = ma50.iloc[-1] if not pd.isna(ma50.iloc[-1]) else current_price
        current_ma200 = ma200.iloc[-1] if not pd.isna(ma200.iloc[-1]) else current_price

        swing_high = close.tail(60).max()
        swing_low = close.tail(60).min()
        swing_range = swing_high - swing_low

        fib_1272 = current_price + (swing_range * 0.272)
        fib_1618 = current_price + (swing_range * 0.618)

        target_atr3 = current_price + (atr * 3)
        target_atr5 = current_price + (atr * 5)
        target_atr8 = current_price + (atr * 8)

        ath = close.max()
        max_reasonable_target = max(current_ma200 * 2, current_price + (atr * 15))

        all_targets = [
            ('ma50', current_ma50),
            ('ma200', current_ma200),
            ('atr3', target_atr3),
            ('atr5', target_atr5),
            ('atr8', target_atr8),
            ('fib1272', fib_1272),
            ('fib1618', fib_1618),
            ('ath', min(ath, max_reasonable_target))
        ]

        # Filter target yang lebih besar dari harga saat ini
        valid_targets = [t for t in all_targets if t[1] > current_price * 1.01]  # Minimal 1% di atas harga

        if len(valid_targets) == 0:
            # Jika semua target di bawah harga, gunakan ATR-based target yang lebih tinggi
            target_atr10 = current_price + (atr * 10)
            valid_targets = [('atr10', target_atr10)]

        valid_targets.sort(key=lambda x: x[1])

        target_konservatif = valid_targets[0][1]
        target_moderat = valid_targets[len(valid_targets)//2][1] if len(valid_targets) > 1 else valid_targets[0][1]
        target_agresif = valid_targets[-1][1]

        fraction = 5 if current_price < 100 else 10 if current_price < 500 else 25
        target_konservatif = round(target_konservatif / fraction) * fraction
        target_moderat = round(target_moderat / fraction) * fraction
        target_agresif = round(target_agresif / fraction) * fraction
        ath_rounded = round(min(ath, max_reasonable_target) / fraction) * fraction

        return {
            'target_konservatif': int(target_konservatif),
            'target_moderat': int(target_moderat),
            'target_agresif': int(target_agresif),
            'target_ath': int(ath_rounded)
        }

    def calculate_teknikal_score(self, df, price, ma50, ma200) -> Dict:
        """Hitung skor teknikal (0-100)"""
        score = 0
        reasons = []

        if price > ma200:
            score += 30
            reasons.append("above MA200")
        elif price > ma200 * 0.9:
            score += 15
            reasons.append("near MA200")

        dist_to_ma50 = (price / ma50 - 1) * 100
        if -5 <= dist_to_ma50 <= 5:
            score += 20
            reasons.append(f"at MA50")
        elif -10 <= dist_to_ma50 <= 10:
            score += 10
            reasons.append(f"near MA50")

        close = df['Close'].shift(1)
        ma20 = close.rolling(20).mean()
        if ma20.iloc[-1] > ma50:
            score += 25
            reasons.append("uptrend")

        returns = close.pct_change().dropna() * 100
        vol = returns.tail(60).std()
        if vol < 2.5:
            score += 25
            reasons.append("low vol")
        elif vol < 4:
            score += 15
            reasons.append("med vol")

        return {'score': score, 'reasons': ', '.join(reasons)}

    def calculate_risk_score(self, df, atr, price, spread) -> float:
        """Hitung skor risk (0-100, semakin tinggi semakin aman)"""
        score = 0

        atr_pct = (atr / price) * 100
        if atr_pct < 2:
            score += 40
        elif atr_pct < 4:
            score += 25
        elif atr_pct < 6:
            score += 10

        if spread < 2:
            score += 30
        elif spread < 4:
            score += 20
        elif spread < 6:
            score += 10

        volume = df['Volume'].tail(60)
        vol_cv = volume.std() / volume.mean()
        if vol_cv < 0.5:
            score += 30
        elif vol_cv < 1:
            score += 15

        return score

    def get_signal(self, symbol: str, df: pd.DataFrame):
        try:
            quality_filter = self.modal_adapter.get_quality_filter()
            if quality_filter == "QUALITY" and symbol not in self.quality_stocks:
                return None
            elif quality_filter == "LQ45" and symbol not in QUALITY_STOCKS[:45]:
                return None

            min_history = self.modal_adapter.get_filter_min_history()
            if len(df) < min_history:
                return None

            df_logic = df.shift(1).copy()
            close_logic = df_logic['Close']
            ma200_logic = close_logic.rolling(window=200).mean()
            ma50_logic = close_logic.rolling(window=50).mean()

            if len(ma200_logic.dropna()) < 1 or len(ma50_logic.dropna()) < 1:
                return None

            current_price_logic = float(close_logic.iloc[-1])
            current_ma200 = float(ma200_logic.iloc[-1]) if not pd.isna(ma200_logic.iloc[-1]) else current_price_logic
            current_ma50 = float(ma50_logic.iloc[-1]) if not pd.isna(ma50_logic.iloc[-1]) else current_price_logic

            current_price_real = float(df['Close'].iloc[-1])

            base_entry_tolerance = self.modal_adapter.get_entry_tolerance()
            regime_params = self.apply_regime_parameters({
                'entry_tolerance': base_entry_tolerance,
                'trend_tolerance': self.modal_adapter.get_trend_tolerance()
            })

            trend_tolerance = regime_params.get('trend_tolerance', self.modal_adapter.get_trend_tolerance())
            entry_tolerance = regime_params.get('entry_tolerance', base_entry_tolerance)

            if current_price_logic < current_ma200 * trend_tolerance:
                return None

            price_to_ma50 = (current_price_logic / current_ma50 - 1) * 100

            if price_to_ma50 > entry_tolerance:
                return None

            if price_to_ma50 < -entry_tolerance * 1.5:
                return None

            min_price, max_price = self.modal_adapter.get_filter_harga()
            if current_price_logic < min_price or current_price_logic > max_price:
                return None

            min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
            last_volume = int(df_logic['Volume'].iloc[-1])
            avg_volume_20 = df_logic['Volume'].tail(20).mean()

            if last_volume < min_vol or avg_volume_20 < min_avg_vol:
                return None

            spread = calculate_spread_pct(df)
            max_spread = self.modal_adapter.get_filter_spread()
            if spread > max_spread:
                return None

            atr = self.risk_manager.calculate_atr_in_rupiah(df)

            # ===== DIVIDEN ANALYSIS =====
            dividend_analysis = {'has_dividend': False, 'total_score': 0, 'display': 'NO_DIVIDEND'}
            if self.warehouse:
                dividend_df = self.warehouse.get_dividends(symbol)
                if dividend_df is not None:
                    dividend_analysis = self.dividend_analyzer.analyze(
                        symbol, dividend_df, current_price_real
                    )

            # ===== TEKNIKAL SCORE =====
            teknikal_score = self.calculate_teknikal_score(
                df, current_price_logic, current_ma50, current_ma200
            )

            # ===== RISK SCORE =====
            risk_score = self.calculate_risk_score(
                df, atr, current_price_logic, spread
            )

            # ===== FINAL SCORE =====
            final_score = (
                teknikal_score['score'] * self.weights['teknikal'] +
                dividend_analysis.get('total_score', 0) * self.weights['dividen'] +
                risk_score * self.weights['risk']
            )

            # ===== LOT CALCULATION =====
            lot, cost, risk_amount = self.risk_manager.calculate_lot(
                current_price_logic, atr, symbol, use_kelly=True
            )

            if lot is None:
                return None

            can_add, reason = self.risk_manager.can_add_position(risk_amount)
            if not can_add:
                return None

            stop_loss = current_ma200 * trend_tolerance * 0.95
            sector = get_sector(symbol)
            risk_pct = (risk_amount / self.risk_manager.modal) * 100
            targets = self.calculate_target_prices(current_price_logic, atr, df)

            # ===== INVESTMENT THESIS =====
            if dividend_analysis.get('has_dividend', False):
                thesis = f"{dividend_analysis['display']}: {dividend_analysis['metrics']['dividend_yield']}% yield"
            else:
                thesis = "GROWTH: pure capital appreciation"

            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(current_price_real),
                'MA50': int(current_ma50),
                'MA200': int(current_ma200),
                'To_MA50': f"{price_to_ma50:.1f}%",
                'Stop_Loss': int(stop_loss),
                'Target_Konservatif': targets['target_konservatif'],
                'Target_Moderat': targets['target_moderat'],
                'Target_Agresif': targets['target_agresif'],
                'Target_ATH': targets['target_ath'],
                'Lot': lot,
                'Cost': int(cost),
                'Risk_Amount': int(risk_amount),
                'Risk_Pct': round(risk_pct, 1),
                'ATR': int(atr),

                # Dividend fields
                'Dividend_Display': dividend_analysis.get('display', 'N/A'),
                'Dividend_Yield': dividend_analysis.get('metrics', {}).get('dividend_yield'),
                'Dividend_Score': dividend_analysis.get('total_score'),
                'Dividend_Category': dividend_analysis.get('category'),
                'Dividend_Years': dividend_analysis.get('metrics', {}).get('consistency_years'),

                # Scoring
                'Teknikal_Score': teknikal_score['score'],
                'Risk_Score': risk_score,
                'Final_Score': round(final_score, 1),

                # Thesis
                'Thesis': thesis,
                'Suitable_For': ', '.join(dividend_analysis.get('suitability', ['All'])),
                'Reasons': teknikal_score['reasons']
            }

        except Exception as e:
            logger.error(f"Error in InvestasiEngine.get_signal for {symbol}: {str(e)}")
            return None

# =============================================================================
# 18. SWING ENGINE (DENGAN REGIME) - DIPERBAIKI
# =============================================================================

class SwingEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher, engine_type='swing')
        self.rsi_period = getattr(config, 'RSI_PERIOD', 14)
        self.ma_short = getattr(config, 'MA_SHORT', 20)
        self.ma_long = getattr(config, 'MA_LONG', 50)
        self.ma200_period = getattr(config, 'MA200_PERIOD', 200)
        self.atr_period = getattr(config, 'ATR_PERIOD', 14)
        self.sl_multiplier = getattr(config, 'SL_MULTIPLIER', 1.5)
        self.tp_multiplier = getattr(config, 'TP_MULTIPLIER', 2.5)
        self.volume_boost = getattr(config, 'VOLUME_BOOST', 1.5)
        self.base_min_rr = getattr(config, 'MIN_RR', 1.0)
        self.min_ev_pct = getattr(config, 'MIN_EV_PCT', 2.0)

        self.modal_adapter = ModalAdapter(config.MODAL, self.engine_type)

    def calculate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']

            delta = close.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.rolling(window=self.rsi_period, min_periods=self.rsi_period).mean()
            avg_loss = loss.rolling(window=self.rsi_period, min_periods=self.rsi_period).mean()
            rs = avg_gain / (avg_loss + 1e-6)
            out['RSI'] = 100 - (100 / (1 + rs))

            out['MA20'] = close.rolling(window=self.ma_short, min_periods=self.ma_short).mean()
            out['MA50'] = close.rolling(window=self.ma_long, min_periods=self.ma_long).mean()
            out['MA200'] = close.rolling(window=self.ma200_period, min_periods=self.ma200_period).mean()
            out['MA_Trend'] = (out['MA20'] > out['MA50']).astype(float)

            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.atr_period, min_periods=self.atr_period).mean()

            out['Volume_MA'] = volume.rolling(window=20, min_periods=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)

            out = out.replace([np.inf, -np.inf], np.nan)

            return out.shift(1)

        except Exception as e:
            logger.error(f"Error in calculate_features: {str(e)}")
            return pd.DataFrame()

    def get_signal(self, symbol: str, df: pd.DataFrame):
        try:
            min_history = self.modal_adapter.get_filter_min_history()
            if len(df) < min_history:
                return None

            df_feat = self.calculate_features(df)

            if len(df_feat) < self.ma200_period + 10:
                return None

            latest_logic = df_feat.iloc[-1]
            close_logic = float(latest_logic['Close']) if not pd.isna(latest_logic['Close']) else None
            if close_logic is None:
                return None

            close_real = float(df['Close'].iloc[-1])

            regime_params = self.apply_regime_parameters({
                'min_rr': self.base_min_rr
            })
            min_rr = regime_params.get('min_rr', self.base_min_rr)

            min_price, max_price = self.modal_adapter.get_filter_harga()
            if close_logic < min_price or close_logic > max_price:
                return None

            min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
            last_volume = int(df['Volume'].iloc[-2]) if len(df) > 1 else 0
            avg_volume_20 = df['Volume'].shift(1).tail(20).mean()

            if last_volume < min_vol or (avg_volume_20 and avg_volume_20 < min_avg_vol):
                return None

            spread = calculate_spread_pct(df)
            max_spread = self.modal_adapter.get_filter_spread()
            if spread > max_spread:
                return None

            ma200 = float(latest_logic['MA200']) if not pd.isna(latest_logic['MA200']) else close_logic
            trend_tolerance = self.modal_adapter.get_trend_tolerance()
            if close_logic < ma200 * trend_tolerance:
                return None

            rsi = float(latest_logic['RSI']) if not pd.isna(latest_logic['RSI']) else 50
            volume_ratio = float(latest_logic['Volume_Ratio']) if not pd.isna(latest_logic['Volume_Ratio']) else 1
            ma20 = float(latest_logic['MA20']) if not pd.isna(latest_logic['MA20']) else close_logic
            ma50 = float(latest_logic['MA50']) if not pd.isna(latest_logic['MA50']) else close_logic

            score = 0
            if rsi < 30:
                score += 3
            elif rsi < 40:
                score += 1

            if volume_ratio > self.volume_boost:
                score += 2
            elif volume_ratio > 1.2:
                score += 1

            if ma20 > ma50:
                score += 1

            atr = self.risk_manager.calculate_atr_in_rupiah(df)

            vol_factor = atr / close_logic
            adaptive_sl_multiplier = self.sl_multiplier * (1 + vol_factor * 10)
            adaptive_tp_multiplier = self.tp_multiplier * (1 - vol_factor * 5)

            adaptive_sl_multiplier = min(max(adaptive_sl_multiplier, 1.0), 3.0)
            adaptive_tp_multiplier = max(adaptive_tp_multiplier, 1.0)

            sl = close_logic - (atr * adaptive_sl_multiplier)
            tp = close_logic + (atr * adaptive_tp_multiplier)

            fraction = 5 if close_logic < 100 else 10 if close_logic < 500 else 25
            sl = round(sl / fraction) * fraction
            tp = round(tp / fraction) * fraction

            if sl >= close_logic or tp <= close_logic:
                return None

            risk = close_logic - sl
            reward = tp - close_logic
            rr = reward / risk

            if rr < min_rr:
                return None

            prob_up = 0.5 + (score * 0.02)
            prob_up = min(prob_up, 0.7)
            expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
            ev_pct = (expected_value / close_logic) * 100

            if ev_pct < self.min_ev_pct:
                return None

            lot, cost, risk_amount = self.risk_manager.calculate_lot(
                close_logic, atr, symbol, use_kelly=True
            )

            if lot is None:
                return None

            can_add, reason = self.risk_manager.can_add_position(risk_amount)
            if not can_add:
                return None

            sector = get_sector(symbol)
            risk_pct = (risk_amount / self.risk_manager.modal) * 100

            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(close_real),
                'RSI': round(rsi, 1),
                'Stop_Loss': int(sl),
                'Take_Profit': int(tp),
                'R/R': round(rr, 2),
                'Prob_Up': round(prob_up, 3),
                'EV_Pct': round(ev_pct, 2),
                'Score': score,
                'ATR': int(atr),
                'Lot': lot,
                'Cost': int(cost),
                'Risk_Amount': int(risk_amount),
                'Risk_Pct': round(risk_pct, 1),
                'Volume': f"{volume_ratio:.1f}x",
                'Reasons': f"RSI {rsi:.0f}, Vol {volume_ratio:.1f}x",
                'Chart': "BULLISH" if ma20 > ma50 else "NETRAL"
            }

        except Exception as e:
            logger.error(f"Error in SwingEngine.get_signal for {symbol}: {str(e)}")
            return None

# =============================================================================
# 19. INTRADAY LIQUID ENGINE (DENGAN REGIME) - DIPERBAIKI
# =============================================================================

class IntradayLiquidEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher, engine_type='intraday_liquid')
        self.breakout_period = getattr(config, 'BREAKOUT_PERIOD', 10)
        self.ma_short = getattr(config, 'MA_SHORT', 20)
        self.ma_long = getattr(config, 'MA_LONG', 50)
        self.atr_period = getattr(config, 'ATR_PERIOD', 14)
        self.volume_breakout = getattr(config, 'VOLUME_BREAKOUT', 1.2)
        self.sl_multiplier = getattr(config, 'SL_MULTIPLIER', 0.8)
        self.tp_multiplier = getattr(config, 'TP_MULTIPLIER', 1.2)
        self.base_min_rr = getattr(config, 'MIN_RR', 1.0)
        self.min_ev_pct = getattr(config, 'MIN_EV_PCT', 2.0)

        self.modal_adapter = ModalAdapter(config.MODAL, self.engine_type)

    def calculate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']
            open_price = out['Open']

            out['Highest_High'] = high.rolling(window=self.breakout_period).max()

            out['MA_Short'] = close.rolling(window=self.ma_short).mean()
            out['MA_Long'] = close.rolling(window=self.ma_long).mean()
            out['MA_Alignment'] = (out['MA_Short'] > out['MA_Long']).astype(int)

            out['Volume_MA'] = volume.rolling(window=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)

            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.atr_period).mean()

            out['Body'] = abs(close - open_price)
            out['Range'] = high - low
            out['Body_Ratio'] = out['Body'] / (out['Range'] + 1e-6)

            out = out.replace([np.inf, -np.inf], np.nan)

            return out.shift(1)

        except Exception as e:
            logger.error(f"Error in calculate_features: {str(e)}")
            return pd.DataFrame()

    def check_breakout(self, row: pd.Series) -> bool:
        if pd.isna(row['Highest_High']) or pd.isna(row['Close']):
            return False
        if row['Close'] <= row['Highest_High']:
            return False
        if row['Volume_Ratio'] < self.volume_breakout:
            return False
        if row['MA_Alignment'] != 1:
            return False
        return True

    def get_signal(self, symbol: str, df: pd.DataFrame):
        try:
            min_history = self.modal_adapter.get_filter_min_history()
            if len(df) < min_history:
                return None

            df_feat = self.calculate_features(df)

            if len(df_feat) < 30:
                return None

            latest_logic = df_feat.iloc[-1]
            close_logic = float(latest_logic['Close']) if not pd.isna(latest_logic['Close']) else None
            if close_logic is None:
                return None

            close_real = float(df['Close'].iloc[-1])

            regime_params = self.apply_regime_parameters({
                'min_rr': self.base_min_rr
            })
            min_rr = regime_params.get('min_rr', self.base_min_rr)

            min_price, max_price = self.modal_adapter.get_filter_harga()
            if close_logic < min_price or close_logic > max_price:
                return None

            min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
            last_volume = int(df['Volume'].iloc[-2]) if len(df) > 1 else 0
            avg_volume_20 = df['Volume'].shift(1).tail(20).mean()

            if last_volume < min_vol or (avg_volume_20 and avg_volume_20 < min_avg_vol):
                return None

            spread = calculate_spread_pct(df)
            max_spread = self.modal_adapter.get_filter_spread()
            if spread > max_spread:
                return None

            if not self.check_breakout(latest_logic):
                return None

            atr = self.risk_manager.calculate_atr_in_rupiah(df)
            atr = max(atr, close_logic * 0.005)

            vol_factor = atr / close_logic
            adaptive_sl_multiplier = self.sl_multiplier * (1 + vol_factor * 5)
            adaptive_tp_multiplier = self.tp_multiplier * (1 - vol_factor * 3)

            adaptive_sl_multiplier = min(max(adaptive_sl_multiplier, 0.5), 2.0)
            adaptive_tp_multiplier = max(adaptive_tp_multiplier, 0.8)

            sl = close_logic - (atr * adaptive_sl_multiplier)
            tp = close_logic + (atr * adaptive_tp_multiplier)

            fraction = 5 if close_logic < 100 else 10 if close_logic < 500 else 25
            sl = round(sl / fraction) * fraction
            tp = round(tp / fraction) * fraction

            if sl >= close_logic or tp <= close_logic:
                return None

            risk = close_logic - sl
            reward = tp - close_logic
            rr = reward / risk

            if rr < min_rr:
                return None

            score = 5
            if latest_logic['Volume_Ratio'] > 1.5:
                score += 1
            if latest_logic['Body_Ratio'] > 0.6:
                score += 1

            prob_up = 0.5 + (score * 0.02)
            prob_up = min(prob_up, 0.7)
            expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
            ev_pct = (expected_value / close_logic) * 100

            if ev_pct < self.min_ev_pct:
                return None

            lot, cost, risk_amount = self.risk_manager.calculate_lot(
                close_logic, atr, symbol, use_kelly=True
            )

            if lot is None:
                return None

            can_add, reason = self.risk_manager.can_add_position(risk_amount)
            if not can_add:
                return None

            sector = get_sector(symbol)
            risk_pct = (risk_amount / self.risk_manager.modal) * 100

            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(close_real),
                'Stop_Loss': int(sl),
                'Take_Profit': int(tp),
                'R/R': round(rr, 2),
                'Prob_Up': round(prob_up, 3),
                'EV_Pct': round(ev_pct, 2),
                'Score': score,
                'ATR': int(atr),
                'Lot': lot,
                'Cost': int(cost),
                'Risk_Amount': int(risk_amount),
                'Risk_Pct': round(risk_pct, 1),
                'Volume': f"{latest_logic['Volume_Ratio']:.1f}x",
                'Reasons': f"Breakout {self.breakout_period}, Vol {latest_logic['Volume_Ratio']:.1f}x",
                'Chart': "BULLISH"
            }

        except Exception as e:
            logger.error(f"Error in IntradayLiquidEngine.get_signal for {symbol}: {str(e)}")
            return None

# =============================================================================
# 20. INTRADAY GORENGAN ENGINE (DENGAN REGIME) - DIPERBAIKI
# =============================================================================

class IntradayGorenganEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher, engine_type='intraday_gorengan')
        self.atr_period = getattr(config, 'ATR_PERIOD', 14)
        self.min_volume_spike = getattr(config, 'MIN_VOLUME_SPIKE', 2.0)
        self.base_min_rr = getattr(config, 'MIN_RR', 1.0)
        self.min_ev_pct = getattr(config, 'MIN_EV_PCT', 1.5)

        self.modal_adapter = ModalAdapter(config.MODAL, self.engine_type)

    def calculate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']
            open_price = out['Open']

            out['Highest_High_5'] = high.rolling(window=5).max()

            out['Volume_MA'] = volume.rolling(window=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)
            out['Turnover'] = close * volume

            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.atr_period).mean()

            out['Body'] = abs(close - open_price)
            out['Range'] = high - low
            out['Body_Ratio'] = out['Body'] / (out['Range'] + 1e-6)
            out['Day_Change'] = (close / open_price - 1) * 100

            out = out.replace([np.inf, -np.inf], np.nan)

            return out.shift(1)

        except Exception as e:
            logger.error(f"Error in calculate_features: {str(e)}")
            return pd.DataFrame()

    def check_breakout(self, row: pd.Series, modal: float) -> bool:
        if pd.isna(row['Highest_High_5']) or pd.isna(row['Close']):
            return False
        if row['Close'] <= row['Highest_High_5']:
            return False
        if row['Volume_Ratio'] < self.min_volume_spike:
            return False
        min_turnover = modal * 3
        if row['Turnover'] < min_turnover:
            return False
        if row['Day_Change'] > 20:
            return False
        return True

    def get_signal(self, symbol: str, df: pd.DataFrame):
        try:
            min_history = self.modal_adapter.get_filter_min_history()
            if len(df) < min_history:
                return None

            df_feat = self.calculate_features(df)

            if len(df_feat) < 30:
                return None

            latest_logic = df_feat.iloc[-1]
            close_logic = float(latest_logic['Close']) if not pd.isna(latest_logic['Close']) else None
            if close_logic is None:
                return None

            close_real = float(df['Close'].iloc[-1])

            regime_params = self.apply_regime_parameters({
                'min_rr': self.base_min_rr
            })
            min_rr = regime_params.get('min_rr', self.base_min_rr)

            min_price, max_price = self.modal_adapter.get_filter_harga()
            if close_logic < min_price or close_logic > max_price:
                return None

            min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
            last_volume = int(df['Volume'].iloc[-2]) if len(df) > 1 else 0
            avg_volume_20 = df['Volume'].shift(1).tail(20).mean()

            if last_volume < min_vol or (avg_volume_20 and avg_volume_20 < min_avg_vol):
                return None

            spread = calculate_spread_pct(df)
            max_spread = self.modal_adapter.get_filter_spread()
            if spread > max_spread:
                return None

            if not self.check_breakout(latest_logic, self.risk_manager.modal):
                return None

            atr = self.risk_manager.calculate_atr_in_rupiah(df)
            atr = max(atr, close_logic * 0.01)

            vol_factor = atr / close_logic
            adaptive_sl_multiplier = 1.0 * (1 + vol_factor * 3)
            adaptive_tp_multiplier = 1.5 * (1 - vol_factor * 2)

            adaptive_sl_multiplier = min(max(adaptive_sl_multiplier, 0.8), 2.0)
            adaptive_tp_multiplier = max(adaptive_tp_multiplier, 1.0)

            sl = close_logic - (atr * adaptive_sl_multiplier)
            tp = close_logic + (atr * adaptive_tp_multiplier)

            fraction = 5 if close_logic < 100 else 10 if close_logic < 500 else 25
            sl = round(sl / fraction) * fraction
            tp = round(tp / fraction) * fraction

            if sl >= close_logic or tp <= close_logic:
                return None

            risk = close_logic - sl
            reward = tp - close_logic
            rr = reward / risk

            if rr < min_rr:
                return None

            score = 5
            if latest_logic['Volume_Ratio'] > 3:
                score += 2
            elif latest_logic['Volume_Ratio'] > 2:
                score += 1
            if latest_logic['Body_Ratio'] > 0.7:
                score += 1

            prob_up = 0.5 + (score * 0.02)
            prob_up = min(prob_up, 0.7)
            expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
            ev_pct = (expected_value / close_logic) * 100

            if ev_pct < self.min_ev_pct:
                return None

            lot, cost, risk_amount = self.risk_manager.calculate_lot(
                close_logic, atr, symbol, use_kelly=True
            )

            if lot is None:
                return None

            can_add, reason = self.risk_manager.can_add_position(risk_amount)
            if not can_add:
                return None

            sector = get_sector(symbol)
            risk_pct = (risk_amount / self.risk_manager.modal) * 100

            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(close_real),
                'Stop_Loss': int(sl),
                'Take_Profit': int(tp),
                'R/R': round(rr, 2),
                'Prob_Up': round(prob_up, 3),
                'EV_Pct': round(ev_pct, 2),
                'Score': score,
                'ATR': int(atr),
                'Volume_Spike': f"{latest_logic['Volume_Ratio']:.1f}x",
                'Turnover': f"Rp {latest_logic['Turnover']/1e6:.0f}Jt",
                'Lot': lot,
                'Cost': int(cost),
                'Risk_Amount': int(risk_amount),
                'Risk_Pct': round(risk_pct, 1),
                'Reasons': f"Breakout 5, Vol {latest_logic['Volume_Ratio']:.1f}x",
                'Chart': "BULLISH"
            }

        except Exception as e:
            logger.error(f"Error in IntradayGorenganEngine.get_signal for {symbol}: {str(e)}")
            return None

# =============================================================================
# 21. CONFIGURATION CLASSES (AGGRESSIVE) - DIPERBAIKI
# =============================================================================

class SwingConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'swing'
        self.INTERVAL = "1d"
        self.PERIOD = "6mo"
        self.RSI_PERIOD = 14
        self.MA_SHORT = 20
        self.MA_LONG = 50
        self.MA200_PERIOD = 200
        self.ATR_PERIOD = 14
        self.SL_MULTIPLIER = 1.5
        self.TP_MULTIPLIER = 2.5
        self.VOLUME_BOOST = 1.5
        self.MIN_RR = 1.0
        self.MIN_EV_PCT = 2.0

class IntradayConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'intraday'
        self.INTERVAL = "1h"
        self.PERIOD = "1mo"
        self.BREAKOUT_PERIOD = 10
        self.VOLUME_BREAKOUT = 1.2
        self.MA_SHORT = 20
        self.MA_LONG = 50
        self.ATR_PERIOD = 14
        self.SL_MULTIPLIER = 0.8
        self.TP_MULTIPLIER = 1.2
        self.MIN_RR = 1.0
        self.MIN_EV_PCT = 2.0

class GorenganConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'gorengan'
        self.INTERVAL = "1h"
        self.PERIOD = "1mo"
        self.ATR_PERIOD = 14
        self.MIN_VOLUME_SPIKE = 2.0
        self.MIN_RR = 1.0
        self.MIN_EV_PCT = 1.5

class InvestasiConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'investasi'
        self.INTERVAL = "1d"
        self.PERIOD = "5y"

# =============================================================================
# 22. GOOGLE SHEETS EXPORTER - TIDAK BERUBAH
# =============================================================================

class GoogleSheetsExporter:
    def __init__(self):
        self.initialized = False
        self.spreadsheet = None
        self._url_printed = False

        self.HEADERS = {
            'SWING ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'RSI',
                'Stop Loss', 'Take Profit', 'R/R', 'Prob Up', 'EV %',
                'Score', 'Volume', 'Lot', 'Cost', 'Risk%'
            ],
            'INTRADAY LIQUID ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price',
                'Stop Loss', 'Take Profit', 'R/R', 'Prob Up', 'EV %',
                'Score', 'Volume', 'Lot', 'Cost', 'Risk%'
            ],
            'INTRADAY GORENGAN ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price',
                'Stop Loss', 'Take Profit', 'R/R', 'Prob Up', 'EV %',
                'Score', 'Volume Spike', 'Turnover', 'Lot', 'Cost', 'Risk%'
            ],
            'INVESTASI ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'To MA50',
                'MA50', 'MA200', 'Stop Loss', 'Target Kons', 'Target Mod',
                'Target Agr', 'Dividend', 'Final Score', 'Lot', 'Cost', 'Risk%'
            ]
        }

    def _init_sheets(self):
        try:
            auth.authenticate_user()
            creds, _ = default()
            self.gc = gspread.authorize(creds)
            self.initialized = True
            print("✅ Berhasil konek ke Google Sheets")
            return True
        except Exception as e:
            print(f"❌ Gagal konek ke Google Sheets: {e}")
            return False

    def _get_or_create_spreadsheet(self, sheet_name):
        try:
            self.spreadsheet = self.gc.open(sheet_name)
            print(f"✅ Membuka spreadsheet: {sheet_name}")
        except:
            self.spreadsheet = self.gc.create(sheet_name)
            print(f"✅ Membuat spreadsheet baru: {sheet_name}")

            try:
                email = input("\n📧 Share spreadsheet dengan email (optional, tekan Enter untuk skip): ").strip()
                if email:
                    self.spreadsheet.share(email, perm_type='user', role='writer')
                    print(f"✅ Shared with {email}")
            except:
                pass

        return self.spreadsheet

    def _get_or_create_engine_sheet(self, engine_name):
        sheet_title = engine_name.replace(' ', '_').replace('ENGINE', '').strip()[:100]

        try:
            worksheet = self.spreadsheet.worksheet(sheet_title)
            print(f"✅ Membuka sheet: {sheet_title}")

            existing_headers = worksheet.row_values(1)
            if not existing_headers:
                headers = self.HEADERS.get(engine_name, ['Date', 'Timestamp', 'Symbol', 'Signal'])
                worksheet.append_row(headers)

        except:
            worksheet = self.spreadsheet.add_worksheet(
                title=sheet_title,
                rows=10000,
                cols=30
            )
            print(f"✅ Membuat sheet baru: {sheet_title}")

            headers = self.HEADERS.get(engine_name, ['Date', 'Timestamp', 'Symbol', 'Signal'])
            worksheet.append_row(headers)

        return worksheet

    def _format_signal_row(self, signal, engine_name):
        today = datetime.now().strftime('%Y-%m-%d')
        timestamp = datetime.now().strftime('%H:%M:%S')

        if engine_name == 'SWING ENGINE':
            return [
                today, timestamp,
                signal['Symbol'], signal['Sector'], signal['Price'],
                signal.get('RSI', '-'), signal['Stop_Loss'], signal['Take_Profit'],
                signal['R/R'], f"{signal['Prob_Up']:.0%}", signal['EV_Pct'],
                signal['Score'], signal['Volume'],
                signal['Lot'], signal['Cost'], signal['Risk_Pct']
            ]

        elif engine_name == 'INTRADAY LIQUID ENGINE':
            return [
                today, timestamp,
                signal['Symbol'], signal['Sector'], signal['Price'],
                signal['Stop_Loss'], signal['Take_Profit'],
                signal['R/R'], f"{signal['Prob_Up']:.0%}", signal['EV_Pct'],
                signal['Score'], signal['Volume'],
                signal['Lot'], signal['Cost'], signal['Risk_Pct']
            ]

        elif engine_name == 'INTRADAY GORENGAN ENGINE':
            return [
                today, timestamp,
                signal['Symbol'], signal['Sector'], signal['Price'],
                signal['Stop_Loss'], signal['Take_Profit'],
                signal['R/R'], f"{signal['Prob_Up']:.0%}", signal['EV_Pct'],
                signal['Score'], signal['Volume_Spike'], signal['Turnover'],
                signal['Lot'], signal['Cost'], signal['Risk_Pct']
            ]

        elif engine_name == 'INVESTASI ENGINE':
            return [
                today, timestamp,
                signal['Symbol'], signal['Sector'], signal['Price'],
                signal['To_MA50'], signal['MA50'], signal['MA200'],
                signal['Stop_Loss'], signal['Target_Konservatif'],
                signal['Target_Moderat'], signal['Target_Agresif'],
                signal.get('Dividend_Display', 'N/A'),
                signal.get('Final_Score', 'N/A'),
                signal['Lot'], signal['Cost'], signal['Risk_Pct']
            ]

        else:
            return [today, timestamp, signal['Symbol'], str(signal)]

    def ensure_spreadsheet_exists(self):
        if not self.initialized:
            if not self._init_sheets():
                return False

        if not self.spreadsheet:
            sheet_name = "IDX_Scanner_All_Engines"
            self._get_or_create_spreadsheet(sheet_name)

        if not self._url_printed:
            print(f"\n📊 Google Sheets URL: {self.spreadsheet.url}")
            self._url_printed = True

        return True

    def export_signals(self, signals, engine_name, modal):
        if not signals:
            print(f"ℹ️ Tidak ada sinyal untuk {engine_name}")
            return False

        if not self.initialized:
            if not self._init_sheets():
                return False

        try:
            if not self.spreadsheet:
                sheet_name = "IDX_Scanner_All_Engines"
                self._get_or_create_spreadsheet(sheet_name)

            worksheet = self._get_or_create_engine_sheet(engine_name)

            for signal in signals:
                row = self._format_signal_row(signal, engine_name)
                worksheet.append_row(row)

            print(f"✅ Berhasil export {len(signals)} sinyal ke {engine_name} sheet")

            if not self._url_printed:
                print(f"\n📊 Google Sheets URL: {self.spreadsheet.url}")
                self._url_printed = True

            return True

        except Exception as e:
            print(f"❌ Gagal export ke Google Sheets: {e}")
            return False

# =============================================================================
# 23. PORTFOLIO GUIDE UNTUK INVESTASI ENGINE - DIPERBAIKI
# =============================================================================

def print_investasi_portfolio_guide(signals, modal, risk_manager,
                                   portfolio_risk_calculator=None, price_data=None):
    """
    Panduan portfolio khusus untuk investasi engine
    AMAN TERHADAP NONE VALUES
    """
    if not signals:
        return

    # Filter sinyal dengan skor > 50
    qualified_signals = [s for s in signals if s.get('Final_Score', 0) > 50]

    if len(qualified_signals) < 3:
        print("\n⚠️  KURANG SINYAL BERKUALITAS (Final Score > 50)")
        print("   Menggunakan semua sinyal dengan peringatan...")
        qualified_signals = signals

    sorted_signals = sorted(qualified_signals,
                           key=lambda x: x.get('Final_Score', 0),
                           reverse=True)

    print("\n" + "="*120)
    print("📊 INVESTASI ENGINE - PORTOFOLIO REKOMENDASI")
    print("="*120)
    print(f"Modal: Rp {modal:,}")
    print(f"Risk per trade: {risk_manager.risk_per_trade_pct:.2f}% (Rp {risk_manager.risk_per_trade_rp:,.0f})")
    print(f"Horizon: 1-3 tahun | Fokus: Dividen + Capital Appreciation")
    print("-"*120)

    dividend_stocks = []
    growth_stocks = []

    for s in sorted_signals:
        # AMAN: cek None sebelum 'in' operation
        category = s.get('Dividend_Category', '')
        if category is not None and 'DIVIDEND' in str(category):
            dividend_stocks.append(s)
        else:
            growth_stocks.append(s)

    if len(dividend_stocks) >= 2:
        top_3 = dividend_stocks[:2] + growth_stocks[:1]
        portfolio_type = "INCOME FOCUSED (2 Dividen + 1 Growth)"
    else:
        top_3 = dividend_stocks[:1] + growth_stocks[:2]
        portfolio_type = "GROWTH FOCUSED (1 Dividen + 2 Growth)"

    print(f"\n📈 PORTOFOLIO TYPE: {portfolio_type}")
    print("-"*120)

    total_dividend_yield = 0
    dividend_count = 0
    total_cost = 0

    for i, signal in enumerate(top_3, 1):
        cost = signal.get('Cost', 0)
        div_yield = signal.get('Dividend_Yield', 0)
        # AMAN: handle None
        if div_yield is not None and div_yield != 'N/A':
            try:
                div_yield = float(div_yield)
            except:
                div_yield = 0
        else:
            div_yield = 0

        print(f"\n{i}. {signal['Symbol']} - {signal.get('Dividend_Display', 'GROWTH')}")
        print(f"   Harga: Rp {signal['Price']:,} | Lot: {signal['Lot']} | Biaya: Rp {cost:,}")
        print(f"   Stop Loss: Rp {signal['Stop_Loss']:,} | Target: {signal.get('Target_Moderat', 'HOLD')}")

        if div_yield > 0:
            print(f"   📊 DIVIDEN: {div_yield}% | Konsisten: {signal.get('Dividend_Years', 0)} tahun")
            total_dividend_yield += div_yield
            dividend_count += 1

        print(f"   💡 THESIS: {signal.get('Thesis', '-')}")
        print(f"   🎯 Suitable for: {signal.get('Suitable_For', 'All Investors')}")
        print(f"   🏆 Final Score: {signal.get('Final_Score', 0)}")

        total_cost += cost

    if dividend_count > 0:
        avg_div_yield = total_dividend_yield / dividend_count
        print("\n" + "-"*120)
        print("📊 PORTOFOLIO SUMMARY:")
        print(f"   Total Investasi: Rp {total_cost:,} ({(total_cost/modal)*100:.1f}% modal)")
        print(f"   Rata-rata Dividend Yield: {avg_div_yield:.2f}% per tahun")
        print(f"   Sisa Modal: Rp {modal - total_cost:,}")

    if total_cost > modal * 1.1:
        print("\n⚠️  TOTAL INVESTASI MELEBIHI MODAL! Kurangi lot.")
    elif total_cost < modal * 0.5:
        print("\n💡 Saran: Tambah posisi atau cari sinyal lain untuk alokasi optimal.")

    print("="*120)

# =============================================================================
# 24. PORTFOLIO GUIDE UNTUK ENGINE LAIN - DIPERBAIKI
# =============================================================================

def print_portfolio_guide(signals, modal, risk_manager, portfolio_risk_calculator=None, price_data=None):
    if not signals:
        return

    def calculate_final_score(signal):
        return signal.get('Score', 0)

    sorted_signals = sorted(signals, key=calculate_final_score, reverse=True)
    top_3 = sorted_signals[:3]

    print("\n" + "="*100)
    print("📊 PANDUAN PORTOFOLIO - 3 REKOMENDASI TERBAIK")
    print("="*100)
    print(f"Modal: Rp {modal:,}")
    print(f"Risk per trade: {risk_manager.risk_per_trade_pct:.2f}% (Rp {risk_manager.risk_per_trade_rp:,.0f})")
    print(f"Max portfolio risk: {risk_manager.max_risk_portfolio_pct}% (Rp {risk_manager.max_risk_portfolio_rp:,.0f})")
    print("-"*100)

    total_cost = 0
    total_risk = 0

    if portfolio_risk_calculator and price_data and len(top_3) > 1:
        print("\n📈 CORRELATION CHECK (NORMAL vs STRESSED):")
        for i in range(len(top_3)):
            for j in range(i+1, len(top_3)):
                corr = portfolio_risk_calculator.get_correlation(
                    top_3[i]['Symbol'],
                    top_3[j]['Symbol'],
                    price_data
                )
                stressed_corr = min(corr * 1.3, 0.99)

                if stressed_corr > 0.7:
                    status = "⚠️ HIGH RISK"
                elif stressed_corr > 0.5:
                    status = "🟡 MEDIUM"
                else:
                    status = "✅ LOW"

                print(f"   {top_3[i]['Symbol']} & {top_3[j]['Symbol']}: {corr:.2f} (stress: {stressed_corr:.2f}) - {status}")
        print("-"*100)

    for i, signal in enumerate(top_3, 1):
        risk_amount = signal.get('Risk_Amount', 0)
        risk_pct = (risk_amount / modal) * 100 if modal > 0 else 0
        cost = signal.get('Cost', 0)
        sl = signal.get('Stop_Loss', 0)
        price = signal.get('Price', 0)

        modal_pct = (cost / modal) * 100 if modal > 0 else 0

        print(f"\n{i}. {signal['Symbol']} ({signal.get('Sector', 'OTHER')})")
        print(f"   Harga: Rp {price:,} | Lot: {signal.get('Lot', 1)} | Biaya: Rp {cost:,} ({modal_pct:.1f}% modal)")
        print(f"   Stop Loss: Rp {sl:,} | Risk: Rp {risk_amount:,} ({risk_pct:.1f}% modal)")
        print(f"   Target: {signal.get('Take_Profit', 'HOLD')} | R/R: {signal.get('R/R', '-')}")
        print(f"   Alasan: {signal.get('Reasons', '-')}")

        total_cost += cost
        total_risk += risk_amount

    print("\n" + "-"*100)
    print(f"📈 TOTAL 3 POSISI:")
    print(f"   Total Biaya: Rp {total_cost:,} ({(total_cost/modal)*100:.1f}% modal)")
    print(f"   Total Risk: Rp {total_risk:,} ({(total_risk/modal)*100:.1f}% modal)")
    print(f"   Sisa Modal: Rp {modal - total_cost:,}")

    if total_risk > risk_manager.max_risk_portfolio_rp:
        print(f"\n⚠️  PERINGATAN: Total risk melebihi batas {risk_manager.max_risk_portfolio_pct}%!")
        print(f"   Kurangi lot atau pilih 1-2 posisi saja.")
    else:
        print(f"\n✅ Risk masih dalam batas aman (max {risk_manager.max_risk_portfolio_pct}%).")

    print("="*100)

# =============================================================================
# 25. PHASE 2 - BLOCK BOOTSTRAP MONTE CARLO - TIDAK BERUBAH
# =============================================================================

class BlockBootstrapMonteCarlo:
    def __init__(self, returns_series: pd.Series):
        self.returns = returns_series.dropna()
        self.block_size = self._calculate_optimal_block_size()

    def _calculate_optimal_block_size(self) -> int:
        if len(self.returns) < 100:
            return 20

        try:
            for lag in range(1, min(100, len(self.returns)//4)):
                acf = self.returns.autocorr(lag=lag)
                if not pd.isna(acf):
                    if acf < 0.5 and lag > 5:
                        half_life = lag
                        break
            else:
                half_life = 20

            block_size = min(60, max(10, half_life * 2))
            return block_size

        except Exception as e:
            logger.error(f"Error calculating optimal block size: {str(e)}")
            return 20

    def _create_blocks(self) -> List[np.ndarray]:
        blocks = []
        step = max(1, self.block_size // 2)

        for i in range(0, len(self.returns) - self.block_size + 1, step):
            block = self.returns.iloc[i:i+self.block_size].values
            blocks.append(block)

        return blocks

    def run_simulation(
        self,
        initial_capital: float,
        n_simulations: int = 1000,
        n_trades: int = 100,
        risk_per_trade_pct: float = 3.0
    ) -> Dict:
        blocks = self._create_blocks()

        if not blocks:
            return {}

        results = []

        for sim in range(n_simulations):
            n_blocks_needed = (n_trades + self.block_size - 1) // self.block_size
            sampled_blocks = random.choices(blocks, k=n_blocks_needed)
            sampled_returns = np.concatenate(sampled_blocks)[:n_trades]

            equity = initial_capital
            peak = equity
            max_dd = 0
            loss_streak = 0
            trades_done = 0

            for r in sampled_returns:
                trade_return = equity * (r / 100)
                equity += trade_return
                trades_done += 1

                if equity > peak:
                    peak = equity
                else:
                    dd = (peak - equity) / peak * 100
                    if dd > max_dd:
                        max_dd = dd

                if r < 0:
                    loss_streak += 1
                    if loss_streak >= 5:
                        break
                else:
                    loss_streak = 0

                if equity < initial_capital * 0.3:
                    break

            total_return_pct = (equity / initial_capital - 1) * 100
            results.append({
                'return': total_return_pct,
                'max_dd': max_dd,
                'trades': trades_done
            })

        returns = np.array([r['return'] for r in results])
        max_dds = np.array([r['max_dd'] for r in results])
        trades_counts = np.array([r['trades'] for r in results])

        summary = {
            'n_simulations': n_simulations,
            'n_trades_max': n_trades,
            'block_size': self.block_size,
            'avg_trades_done': np.mean(trades_counts),
            'mean_return': np.mean(returns),
            'median_return': np.median(returns),
            'std_return': np.std(returns),
            'percentile_5': np.percentile(returns, 5),
            'percentile_25': np.percentile(returns, 25),
            'percentile_75': np.percentile(returns, 75),
            'percentile_95': np.percentile(returns, 95),
            'min_return': np.min(returns),
            'max_return': np.max(returns),
            'avg_max_dd': np.mean(max_dds),
            'max_dd_95': np.percentile(max_dds, 95),
            'probability_profit': np.mean(returns > 0) * 100,
            'probability_2x': np.mean(returns > 100) * 100,
            'probability_50pct_loss': np.mean(returns < -50) * 100,
            'expected_return_per_trade': np.mean(returns) / np.mean(trades_counts) if np.mean(trades_counts) > 0 else 0
        }

        self.results = results
        self.summary = summary
        self.returns_array = returns

        return summary

    def print_results(self) -> None:
        if not hasattr(self, 'summary'):
            print("\n📊 No simulation results")
            return

        print("\n" + "="*100)
        print(f"🎲 BLOCK BOOTSTRAP MONTE CARLO (Block Size: {self.summary['block_size']} days)")
        print("="*100)
        print(f"Number of simulations: {self.summary['n_simulations']:,}")
        print(f"Max trades per simulation: {self.summary['n_trades_max']}")
        print(f"Rata-rata trades terealisasi: {self.summary['avg_trades_done']:.1f}")
        print(f"Risk per trade: 3.0%")

        print("\n📈 RETURN DISTRIBUTION:")
        print(f"   Mean return: {self.summary['mean_return']:.2f}%")
        print(f"   Median return: {self.summary['median_return']:.2f}%")
        print(f"   Std deviation: {self.summary['std_return']:.2f}%")
        print(f"   Best case (95th percentile): {self.summary['percentile_95']:.2f}%")
        print(f"   Worst case (5th percentile): {self.summary['percentile_5']:.2f}%")

        print("\n📉 DRAWDOWN ANALYSIS:")
        print(f"   Rata-rata max drawdown: {self.summary['avg_max_dd']:.2f}%")
        print(f"   Max drawdown 95th percentile: {self.summary['max_dd_95']:.2f}%")

        print("\n🎯 PROBABILITIES:")
        print(f"   Probability of profit: {self.summary['probability_profit']:.1f}%")
        print(f"   Probability of 2x money: {self.summary['probability_2x']:.1f}%")
        print(f"   Probability of 50% loss: {self.summary['probability_50pct_loss']:.1f}%")

    def plot_distribution(self, save_path: str = None):
        if not hasattr(self, 'returns_array'):
            print("📊 No simulation results to plot")
            return

        plt.figure(figsize=(10, 6))

        plt.hist(self.returns_array, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
        plt.axvline(self.summary['mean_return'], color='red', linestyle='--', linewidth=2,
                    label=f"Mean: {self.summary['mean_return']:.2f}%")
        plt.axvline(self.summary['percentile_5'], color='orange', linestyle=':', linewidth=2,
                    label=f"5%: {self.summary['percentile_5']:.2f}%")
        plt.axvline(self.summary['percentile_95'], color='green', linestyle=':', linewidth=2,
                    label=f"95%: {self.summary['percentile_95']:.2f}%")

        plt.xlabel('Return (%)', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.title(f'Monte Carlo Simulation (Block Size: {self.summary["block_size"]} days)', fontsize=14)
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)

        textstr = f'Mean: {self.summary["mean_return"]:.2f}%\n'
        textstr += f'Std: {self.summary["std_return"]:.2f}%\n'
        textstr += f'Profit Prob: {self.summary["probability_profit"]:.1f}%'

        plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=100, bbox_inches='tight')
            print(f"✅ Chart saved: {save_path}")
        else:
            plt.show()

# =============================================================================
# 26. VALIDATION SUITE - TIDAK BERUBAH
# =============================================================================

class ValidationSuite:
    def __init__(self, engine_class, base_config, data_dict: Dict[str, pd.DataFrame]):
        self.engine_class = engine_class
        self.base_config = base_config
        self.data_dict = data_dict

    def run_all(self, walk_forward_params=None, sensitivity_params=None, monte_carlo_params=None):
        results = {}

        if monte_carlo_params is None:
            monte_carlo_params = {
                'n_simulations': 500,
                'n_trades': 100
            }

        print("\n" + "="*80)
        print("PHASE 2.3: BLOCK BOOTSTRAP MONTE CARLO (OPTIMAL BLOCK SIZE - AGGRESSIVE 3%)")
        print("="*80)

        all_returns = []
        for symbol, df in self.data_dict.items():
            returns = df['Close'].pct_change().dropna() * 100
            all_returns.extend(returns.values[:1000])

        returns_series = pd.Series(all_returns)

        print(f"\n📦 Optimal Block Size Calculation...")
        mc = BlockBootstrapMonteCarlo(returns_series)
        mc_results = mc.run_simulation(
            initial_capital=self.base_config.MODAL,
            n_simulations=monte_carlo_params['n_simulations'],
            n_trades=monte_carlo_params['n_trades'],
            risk_per_trade_pct=3.0
        )
        mc.print_results()
        mc.plot_distribution(save_path=f'monte_carlo_optimal_aggressive.png')
        results['monte_carlo'] = mc_results

        return results

# =============================================================================
# 27. DATA WAREHOUSE (DENGAN ERROR HANDLING AMAN) - DIPERBAIKI (FOLDER DIVIDEN TERPISAH)
# =============================================================================

class DataWarehouse:
    """Menyimpan data historis LENGKAP untuk semua saham - DENGAN LOGGING"""

    # DATA WAREHOUSE UTAMA TIDAK BERUBAH - AMAN

    def __init__(self, warehouse_dir: str = 'data_warehouse', min_days: int = 400):
        self.warehouse_dir = warehouse_dir
        self.min_days = min_days
        os.makedirs(warehouse_dir, exist_ok=True)

        # FOLDER TERPISAH UNTUK DIVIDEN - TIDAK MENGGANGGU DATA UTAMA
        self.dividend_dir = f'{warehouse_dir}/dividends'
        os.makedirs(self.dividend_dir, exist_ok=True)

        self.stats = {
            'total_symbols': 0,
            'downloaded': 0,
            'failed': 0,
            'cached': 0,
            'filtered_min_days': 0,
            'corrupt_files': 0
        }

    def download_complete_history(
        self,
        symbols: List[str],
        start_date: str = '2018-01-01',
        end_date: str = '2026-12-31',
        force_refresh: bool = False
    ) -> Dict[str, pd.DataFrame]:
        print("\n" + "="*80)
        print("🗄️  DATA WAREHOUSE - DOWNLOAD HISTORIS LENGKAP")
        print("="*80)
        print(f"Periode: {start_date} hingga {end_date}")
        print(f"Total saham: {len(symbols)}")
        print(f"Minimal hari: {self.min_days} (saham dengan data kurang akan difilter)")

        results = {}
        self.stats['total_symbols'] = len(symbols)

        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)

        for i, symbol in enumerate(symbols):
            cache_file = f"{self.warehouse_dir}/{symbol}_full.parquet"

            if os.path.exists(cache_file) and not force_refresh:
                try:
                    df = pd.read_parquet(cache_file)

                    if len(df) >= self.min_days and df.index[0] <= start_dt and df.index[-1] >= end_dt:
                        results[symbol] = df
                        self.stats['cached'] += 1
                    else:
                        self.stats['filtered_min_days'] += 1

                    if (i + 1) % 50 == 0:
                        print(f"   Progress: {i+1}/{len(symbols)} - {len(results)} dimuat (cache)")
                    continue
                except Exception as e:
                    logger.error(f"Corrupt cache file for {symbol}: {str(e)}")
                    self.stats['corrupt_files'] += 1

            try:
                ticker = f"{symbol}.JK"
                print(f"   Downloading {symbol} ({i+1}/{len(symbols)})...", end=" ")

                df = yf.download(
                    ticker,
                    start=start_date,
                    end=end_date,
                    interval="1d",
                    auto_adjust=True,
                    progress=False,
                    timeout=30
                )

                time.sleep(0.5)

                if df.empty:
                    print("❌ GAGAL (data kosong)")
                    self.stats['failed'] += 1
                    logger.warning(f"Empty data for {symbol}")
                    continue

                df = normalize_columns(df)

                if len(df) < self.min_days:
                    print(f"❌ GAGAL (hanya {len(df)} hari, minimal {self.min_days})")
                    self.stats['filtered_min_days'] += 1
                    continue

                df.to_parquet(cache_file)
                results[symbol] = df
                self.stats['downloaded'] += 1
                print(f"✅ {len(df)} hari")

            except Exception as e:
                print(f"❌ ERROR: {str(e)[:50]}")
                self.stats['failed'] += 1
                logger.error(f"Error downloading {symbol}: {str(e)}")
                time.sleep(1)

        print("\n" + "="*80)
        print("📊 RINGKASAN DATA WAREHOUSE")
        print("="*80)
        print(f"Total saham: {self.stats['total_symbols']}")
        print(f"Berhasil dimuat: {len(results)}")
        print(f"  - Dari cache: {self.stats['cached']}")
        print(f"  - Download baru: {self.stats['downloaded']}")
        print(f"  - Difilter (< {self.min_days} hari): {self.stats['filtered_min_days']}")
        print(f"  - File corrupt: {self.stats['corrupt_files']}")
        print(f"Gagal: {self.stats['failed']}")
        print("="*80)

        return results

    def get_all_valid_symbols(self) -> List[str]:
        symbols = []
        for f in os.listdir(self.warehouse_dir):
            if f.endswith('_full.parquet'):
                symbol = f.replace('_full.parquet', '')
                try:
                    df = pd.read_parquet(os.path.join(self.warehouse_dir, f))
                    if len(df) >= self.min_days:
                        symbols.append(symbol)
                except Exception as e:
                    logger.error(f"Corrupt file in get_all_valid_symbols: {f} - {str(e)}")
                    continue
        return symbols

    def get_data(self, symbol: str) -> Optional[pd.DataFrame]:
        cache_file = f"{self.warehouse_dir}/{symbol}_full.parquet"

        if not os.path.exists(cache_file):
            return None

        try:
            df = pd.read_parquet(cache_file)
            if len(df) >= self.min_days:
                return df
            return None
        except Exception as e:
            logger.error(f"Error reading {symbol}: {str(e)}")
            return None

    def get_all_data(self, max_symbols: int = None) -> Dict[str, pd.DataFrame]:
        results = {}
        symbols = self.get_all_valid_symbols()

        if max_symbols:
            symbols = symbols[:max_symbols]

        for symbol in symbols:
            df = self.get_data(symbol)
            if df is not None:
                results[symbol] = df

        return results

    def print_warehouse_stats(self) -> None:
        symbols = self.get_all_valid_symbols()

        print("\n" + "="*80)
        print("🗄️  DATA WAREHOUSE STATISTICS")
        print("="*80)
        print(f"Total saham valid (≥{self.min_days} hari): {len(symbols)}")

        if symbols:
            sample = symbols[0]
            df = pd.read_parquet(f"{self.warehouse_dir}/{sample}_full.parquet")
            print(f"Rentang tanggal: {df.index[0].date()} hingga {df.index[-1].date()}")
            print(f"Rata-rata hari per saham: {len(df)} hari")

        print("="*80)

    # ==================== DIVIDEN METHODS (FOLDER TERPISAH) - DIPERBAIKI ====================

    def download_dividend_history(self, symbols: List[str], years_back: int = 10) -> Dict[str, pd.DataFrame]:
        """
        Download historical dividend data untuk semua saham
        DISIMPAN DI FOLDER TERPISAH - TIDAK MENGGANGGU DATA UTAMA
        FIX: Menghapus timezone dari index untuk menghindari error perbandingan
        """
        print("\n" + "="*80)
        print("💰 DOWNLOADING DIVIDEND HISTORY (FOLDER TERPISAH)")
        print("="*80)

        results = {}
        end_date = datetime.now()
        start_date = end_date - timedelta(days=365*years_back)
        start_timestamp = pd.Timestamp(start_date)

        for i, symbol in enumerate(symbols):
            cache_file = f"{self.dividend_dir}/{symbol}_dividends.parquet"

            if os.path.exists(cache_file):
                try:
                    df = pd.read_parquet(cache_file)
                    # FIX: Hapus timezone saat membaca dari cache
                    if df.index.tz is not None:
                        df.index = df.index.tz_localize(None)
                    results[symbol] = df
                    if (i+1) % 50 == 0:
                        print(f"   Progress: {i+1}/{len(symbols)} - {len(results)} from cache")
                    continue
                except Exception as e:
                    logger.error(f"Error reading cache for {symbol}: {str(e)}")
                    # Lanjut download ulang jika cache corrupt

            try:
                ticker = yf.Ticker(f"{symbol}.JK")
                dividends = ticker.dividends

                if dividends is not None and len(dividends) > 0:
                    # Konversi ke DataFrame
                    df = pd.DataFrame(dividends)
                    df.columns = ['Dividend']
                    df.index.name = 'Date'

                    # ===== FIX TIMEZONE ISSUE =====
                    # Hapus timezone dari index jika ada
                    if df.index.tz is not None:
                        df.index = df.index.tz_localize(None)

                    # Filter berdasarkan periode (sekarang aman)
                    df_filtered = df[df.index >= start_timestamp]

                    if len(df_filtered) > 0:
                        df_filtered.to_parquet(cache_file)
                        results[symbol] = df_filtered
                        print(f"   ✅ {symbol}: {len(df_filtered)} dividend records")
                    else:
                        print(f"   ⚠️ {symbol}: no dividends in period")
                else:
                    print(f"   ⚠️ {symbol}: no dividend data")

                time.sleep(0.2)

            except Exception as e:
                print(f"   ❌ {symbol}: {str(e)[:50]}")
                logger.error(f"Error downloading dividend for {symbol}: {str(e)}")

        print("\n" + "="*80)
        print(f"✅ Dividend download complete: {len(results)} symbols with dividends")
        print("="*80)

        return results

    def get_dividends(self, symbol: str) -> Optional[pd.DataFrame]:
        """Ambil data dividen dari folder terpisah"""
        cache_file = f"{self.dividend_dir}/{symbol}_dividends.parquet"
        if os.path.exists(cache_file):
            try:
                df = pd.read_parquet(cache_file)
                # FIX: Hapus timezone saat membaca
                if df.index.tz is not None:
                    df.index = df.index.tz_localize(None)
                return df
            except Exception as e:
                logger.error(f"Error reading dividend for {symbol}: {str(e)}")
                return None
        return None

# =============================================================================
# 28. DATA WAREHOUSE INITIALIZATION - DIPERBAIKI (TIDAK LANGSUNG RETURN)
# =============================================================================

def initialize_data_warehouse():
    print("\n" + "="*80)
    print("🗄️  INISIALISASI DATA WAREHOUSE")
    print("="*80)
    print("1. Download data harga saham historis (jika belum ada)")
    print("2. Download data dividen (opsional, folder terpisah)")
    print("="*80)
    print(f"Total saham: {len(STOCKBIT_UNIVERSE)}")
    print(f"Minimal hari: 400 (saham dengan data kurang akan difilter)")

    warehouse = DataWarehouse(warehouse_dir='data_warehouse', min_days=400)

    # CEK DATA HARGA YANG SUDAH ADA
    existing_data = warehouse.get_all_valid_symbols()
    print(f"\n📊 Data harga yang sudah tersedia: {len(existing_data)} saham")

    # ===== 1. OPSI DOWNLOAD DATA HARGA =====
    print("\n" + "="*80)
    print("📈 DATA HARGA SAHAM")
    print("="*80)

    download_harga = 'n'
    if len(existing_data) >= 400:
        print(f"✅ Data harga sudah mencukupi ({len(existing_data)} saham)")
        download_harga = input("   Tetap download ulang data harga? (y/n): ").strip().lower()
    else:
        print("⚠️  Data harga belum lengkap")
        download_harga = input("   Download data harga? (y/n): ").strip().lower()

    if download_harga == 'y':
        print("\n⚠️  PERINGATAN: Download data harga akan memakan waktu BEBERAPA JAM!")
        confirm = input("Lanjutkan download data harga? (y/n): ").strip().lower()
        if confirm == 'y':
            print("\n📥 Mendownload data harga saham...")
            data = warehouse.download_complete_history(
                symbols=STOCKBIT_UNIVERSE,
                start_date='2018-01-01',
                end_date='2026-12-31'
            )
            print(f"✅ Selesai! Data harga tersedia untuk {len(data)} saham")
        else:
            print("⏩ Lewati download data harga")
    else:
        print("⏩ Menggunakan data harga yang sudah ada")

    # ===== 2. OPSI DOWNLOAD DATA DIVIDEN =====
    print("\n" + "="*80)
    print("💰 DIVIDEN DATA")
    print("="*80)
    print("Data dividen disimpan di folder terpisah:")
    print("   data_warehouse/dividends/")
    print("TIDAK MENGGANGGU data harga saham utama")

    # Cek data dividen yang sudah ada
    import glob
    existing_div = glob.glob(f"{warehouse.dividend_dir}/*_dividends.parquet")
    if existing_div:
        print(f"✅ Data dividen sudah tersedia untuk {len(existing_div)} saham")

    div_confirm = input("\nDownload data dividen untuk SEMUA saham? (y/n): ").strip().lower()

    if div_confirm == 'y':
        print(f"\n📥 Mendownload data dividen untuk {len(STOCKBIT_UNIVERSE)} saham...")
        print("   Proses ini akan memakan waktu ~30-60 menit...")

        dividend_results = warehouse.download_dividend_history(
            symbols=STOCKBIT_UNIVERSE,
            years_back=10
        )

        print(f"\n✅ Selesai! Data dividen tersedia untuk {len(dividend_results)} saham")
        print(f"   Saham tanpa data dividen: {len(STOCKBIT_UNIVERSE) - len(dividend_results)}")
    else:
        print("⏩ Lewati download dividen")

    # Tampilkan ringkasan akhir
    print("\n" + "="*80)
    print("📊 RINGKASAN AKHIR DATA WAREHOUSE")
    print("="*80)

    final_data = warehouse.get_all_valid_symbols()
    print(f"Data harga: {len(final_data)} saham valid (≥{warehouse.min_days} hari)")

    final_div = glob.glob(f"{warehouse.dividend_dir}/*_dividends.parquet")
    print(f"Data dividen: {len(final_div)} saham")

    if final_data:
        sample = final_data[0]
        df = warehouse.get_data(sample)
        if df is not None:
            print(f"Rentang tanggal harga: {df.index[0].date()} hingga {df.index[-1].date()}")

    print("="*80)

    # Kembalikan data harga (jika ada)
    if 'data' in locals():
        return data
    else:
        return warehouse.get_all_data()

# =============================================================================
# 28. PHASE 1 - MAIN PROGRAM (TANPA WALKFORWARD COLLECTOR)
# =============================================================================

def run_phase1(sheets_exporter):
    print("\n" + "="*60)
    print("🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE (AGGRESSIVE)")
    print("   Modal-Adaptive Filters | Risk 3% per trade | Target 15-20% per tahun")
    print("   Export: Google Sheets | Panduan Portfolio 3 Terbaik")
    print("="*60)

    warehouse = DataWarehouse(warehouse_dir='data_warehouse', min_days=400)
    symbols = warehouse.get_all_valid_symbols()

    if not symbols:
        print("\n⚠️  Data warehouse kosong atau tidak ditemukan!")
        print("   Jalankan opsi 3 (Inisialisasi Data Warehouse) terlebih dahulu.")
        return

    print(f"\n📊 Data warehouse siap: {len(symbols)} saham dengan data >= 400 hari")

    print("\nPilih engine trading:")
    print("1. Swing Engine (Mingguan - Mean Reversion) - Modal: Rp 40rb - 5jt")
    print("2. Intraday Liquid (1 jam - Momentum) - Modal: Rp 10rb - 500rb")
    print("3. Intraday Gorengan (1 jam - Early Momentum) - Modal: Rp 10rb - 500rb")
    print("4. Investasi Engine (Quality + Trend - Jangka Panjang) - Modal: Rp 100rb - 1jt")

    while True:
        engine_choice = input("Pilihan (1/2/3/4): ").strip()
        if engine_choice in ['1', '2', '3', '4']:
            break
        print("❌ Pilih 1, 2, 3, atau 4")

    modal_ranges = {
        '1': (40000, 5000000, 'swing'),
        '2': (10000, 500000, 'intraday_liquid'),
        '3': (10000, 500000, 'intraday_gorengan'),
        '4': (100000, 1000000, 'investasi')
    }

    min_modal, max_modal, engine_type = modal_ranges[engine_choice]

    while True:
        try:
            modal_input = input(f"\nModal (Rp {min_modal:,} - {max_modal:,}): ").strip()
            modal = int(modal_input.replace('.', '').replace(',', ''))
            if min_modal <= modal <= max_modal:
                break
            print(f"❌ Modal harus {min_modal:,} - {max_modal:,}")
        except Exception:
            print("❌ Input tidak valid")

    if engine_choice == '1':
        config = SwingConfig(modal)
        engine_name = "SWING ENGINE"
        EngineClass = SwingEngine
        timeframe = '1d'
    elif engine_choice == '2':
        config = IntradayConfig(modal)
        engine_name = "INTRADAY LIQUID ENGINE"
        EngineClass = IntradayLiquidEngine
        timeframe = '1h'
    elif engine_choice == '3':
        config = GorenganConfig(modal)
        engine_name = "INTRADAY GORENGAN ENGINE"
        EngineClass = IntradayGorenganEngine
        timeframe = '1h'
    else:
        config = InvestasiConfig(modal)
        engine_name = "INVESTASI ENGINE"
        EngineClass = InvestasiEngine
        timeframe = '1d'

    modal_adapter = ModalAdapter(modal, engine_type)
    modal_adapter.print_info()

    print("\n🚀 Initializing Phase 1 Components...")

    risk_manager = RiskManager(
        modal=modal,
        risk_per_trade_pct=3.0,
        max_risk_portfolio_pct=15.0,
        max_lot_per_position=10,
        engine_type=engine_type
    )
    print(f"   ✅ Risk Manager: Rp {risk_manager.risk_per_trade_rp:,.0f} risk/trade (3.0%)")

    fee_config = RealisticFeeConfig(liquidity='medium')
    print(f"   ✅ Fee Config: min fee Rp {fee_config.MIN_FEE_PER_TRANSACTION:,} (termasuk VAT 11%)")

    global_fetcher = GlobalIndicesFetcher()
    global_fetcher.fetch_all()
    global_fetcher.print_detailed_report()

    regime_detector = MarketRegimeDetector()
    if 'IHSG' in global_fetcher.data:
        ihsg_data = global_fetcher.data['IHSG']
        current_regime, confidence = regime_detector.detect_regime(ihsg_data)
        regime_detector.print_regime_report()
    else:
        print("\n⚠️  IHSG data tidak tersedia, regime detection skipped")

    delay_simulator = EntryDelaySimulator(max_delay=2)
    print(f"   ✅ Entry Delay Simulator: max delay {delay_simulator.max_delay} days (informational only)")

    portfolio_risk = PortfolioRiskCalculator(lookback_days=60)
    print(f"   ✅ Portfolio Risk Calculator: {portfolio_risk.lookback_days} days lookback with stress testing")

    engine = EngineClass(config, global_fetcher)
    engine.set_risk_manager(risk_manager)
    engine.set_regime_detector(regime_detector)

    if engine_choice == '4':
        engine.set_warehouse(warehouse)

    print("\n" + "="*60)
    print(f"📊 {engine_name} - READY (AGGRESSIVE)")
    print("="*60)
    print(f"Modal: Rp {modal:,}")
    print(f"Risk per trade: Rp {risk_manager.risk_per_trade_rp:,.0f} (3.0% AGGRESSIVE)")
    print(f"Max portfolio risk: Rp {risk_manager.max_risk_portfolio_rp:,.0f} (15.0% AGGRESSIVE)")
    print(f"Min fee: Rp {fee_config.MIN_FEE_PER_TRANSACTION:,}")
    print(f"Universe: {len(symbols)} saham valid dari warehouse")

    print(f"\n📥 Pilih jumlah saham yang akan dianalisis:")
    print(f"   1. Semua ({len(symbols)} saham) - Lebih lama tapi komprehensif")
    print(f"   2. Sample 100 saham - Sedang")
    print(f"   3. Sample 50 saham - Cepat")
    print(f"   4. Sample 20 saham - Sangat cepat")

    while True:
        sample_choice = input("Pilihan (1/2/3/4): ").strip()
        if sample_choice == '1':
            test_symbols = symbols
            break
        elif sample_choice == '2':
            test_symbols = symbols[:100]
            break
        elif sample_choice == '3':
            test_symbols = symbols[:50]
            break
        elif sample_choice == '4':
            test_symbols = symbols[:20]
            break
        else:
            print("❌ Pilih 1, 2, 3, atau 4")

    print(f"\n📥 Mengambil data dari warehouse untuk {len(test_symbols)} saham...")

    stocks_data = {}
    for i, symbol in enumerate(test_symbols):
        df = warehouse.get_data(symbol)
        if df is not None:
            stocks_data[symbol] = df

        if (i + 1) % 50 == 0:
            print(f"   Progress: {i+1}/{len(test_symbols)} - {len(stocks_data)} dimuat")

    print(f"\n✅ Berhasil memuat {len(stocks_data)} saham dari warehouse")

    if stocks_data:
        print(f"\n📊 Menganalisis {len(stocks_data)} saham...")
        signals = []

        for symbol, df in stocks_data.items():
            signal = engine.get_signal(symbol, df)
            if signal:
                signals.append(signal)

        print(f"✅ Ditemukan {len(signals)} sinyal")

        if signals:
            print("\n" + "="*100)
            print("🌍 RINGKASAN PASAR")
            print("="*100)
            market_data = []
            for name in GLOBAL_INDICES.keys():
                mom = global_fetcher.get_momentum(name)
                trend = global_fetcher.get_trend(name)
                price_str = global_fetcher.get_price_str(name)
                market_data.append([name, price_str, f"{mom:+.2f}%", trend])
            print(tabulate(market_data, headers=["Indeks", "Harga", "Momentum", "Trend"], tablefmt="grid"))

            print("\n" + "="*100)
            print(f"📊 {engine_name} - REKOMENDASI (Top 5 dari {len(signals)} sinyal)")
            print("="*100)

            # Urutkan berdasarkan Final Score untuk investasi, atau Score untuk lainnya
            if engine_choice == '4':
                sorted_signals = sorted(signals, key=lambda x: x.get('Final_Score', 0), reverse=True)
            else:
                sorted_signals = sorted(signals, key=lambda x: x.get('Score', 0), reverse=True)

            # Ambil 5 terbaik
            top_5 = sorted_signals[:5]

            if engine_choice == '4':
                display_data = []
                for s in top_5:
                    target_str = f"{s.get('Target_Konservatif', '-')} | {s.get('Target_Moderat', '-')} | {s.get('Target_Agresif', '-')}"
                    display_data.append([
                        s['Symbol'],
                        s['Sector'],
                        f"Rp {s['Price']:,}",
                        f"{s['To_MA50']}",
                        f"Rp {s['MA50']:,}",
                        f"Rp {s['MA200']:,}",
                        f"Rp {s['Stop_Loss']:,}",
                        target_str,
                        s.get('Dividend_Display', 'N/A'),
                        f"{s['Lot']} lot",
                        f"Rp {s['Cost']:,}",
                        s.get('Final_Score', 'N/A')
                    ])

                headers = [
                    "Kode", "Sektor", "Harga", "To MA50", "MA50", "MA200",
                    "Stop Loss", "Target", "Dividen", "Lot", "Biaya", "Score"
                ]
            else:
                display_data = []
                for s in top_5:
                    display_data.append([
                        s['Symbol'],
                        s['Sector'],
                        f"Rp {s['Price']:,}",
                        s.get('RSI', '-'),
                        s.get('Volume', '-'),
                        f"{s['R/R']}",
                        f"{s['EV_Pct']}%",
                        f"{s['Score']}",
                        f"Rp {s['Stop_Loss']:,}",
                        f"Rp {s['Take_Profit']:,}",
                        f"{s['Lot']} lot",
                        f"Rp {s['Cost']:,}"
                    ])

                headers = [
                    "Kode", "Sektor", "Harga", "RSI", "Vol", "R/R",
                    "EV%", "Skor", "SL", "TP", "Lot", "Biaya"
                ]

            print(tabulate(display_data, headers=headers, tablefmt='grid', stralign='left', numalign='center'))

            if engine_choice == '4':
                print_investasi_portfolio_guide(signals, modal, risk_manager, portfolio_risk, stocks_data)
            else:
                print_portfolio_guide(signals, modal, risk_manager, portfolio_risk, stocks_data)

            export_choice = input(f"\n📊 Export {engine_name} signals ke Google Sheets? (y/n): ").strip().lower()
            if export_choice == 'y':
                sheets_exporter.export_signals(top_5, engine_name, modal)  # Export hanya 5 terbaik

        else:
            print("\n❌ Tidak ada sinyal hari ini")
    else:
        print("\n❌ Tidak ada data yang berhasil dimuat dari warehouse")

    print("\n" + "="*60)
    print("✅ PHASE 1 COMPLETE - AGGRESSIVE MODE")
    print("   Risk Management: 3% per trade, 15% portfolio")
    print("="*60)

# =============================================================================
# 29. PHASE 2 - MAIN PROGRAM - TIDAK BERUBAH
# =============================================================================

def run_phase2():
    print("\n" + "="*80)
    print("📊 IDX STOCK SCANNER - PHASE 2 VALIDATION (AGGRESSIVE)")
    print("   Monte Carlo FIXED | Walk-Forward | Sensitivity | Risk 3%")
    print("="*80)

    warehouse = DataWarehouse(warehouse_dir='data_warehouse', min_days=400)
    symbols = warehouse.get_all_valid_symbols()

    if not symbols:
        print("\n⚠️  Data warehouse kosong!")
        print("   Jalankan opsi 3 (Inisialisasi Data Warehouse) terlebih dahulu.")
        return

    print(f"\n📥 Loading data dari warehouse ({len(symbols)} saham tersedia)...")

    print(f"\n📊 Pilih jumlah saham untuk validasi:")
    print(f"   1. Semua ({len(symbols)} saham)")
    print(f"   2. Sample 200 saham")
    print(f"   3. Sample 100 saham")

    while True:
        sample_choice = input("Pilihan (1/2/3): ").strip()
        if sample_choice == '1':
            test_symbols = symbols
            break
        elif sample_choice == '2':
            test_symbols = symbols[:200]
            break
        elif sample_choice == '3':
            test_symbols = symbols[:100]
            break
        else:
            print("❌ Pilih 1, 2, atau 3")

    data_dict = {}
    for i, symbol in enumerate(test_symbols):
        df = warehouse.get_data(symbol)
        if df is not None:
            data_dict[symbol] = df

        if (i+1) % 50 == 0:
            print(f"   Loaded {i+1}/{len(test_symbols)} - {len(data_dict)} valid")

    print(f"\n✅ Loaded {len(data_dict)} stocks dengan data lengkap dari warehouse")

    if not data_dict:
        print("❌ No data loaded. Cannot proceed.")
        return

    print("\nPilih engine untuk divalidasi:")
    print("1. Swing Engine")
    print("2. Intraday Liquid")
    print("3. Intraday Gorengan")
    print("4. Investasi Engine")

    while True:
        choice = input("Pilihan (1/2/3/4): ").strip()
        if choice in ['1', '2', '3', '4']:
            break
        print("❌ Pilih 1-4")

    if choice == '1':
        engine_class = SwingEngine
        config = SwingConfig(modal=100000000)
        engine_name = "SWING ENGINE"
    elif choice == '2':
        engine_class = IntradayLiquidEngine
        config = IntradayConfig(modal=100000000)
        engine_name = "INTRADAY LIQUID ENGINE"
    elif choice == '3':
        engine_class = IntradayGorenganEngine
        config = GorenganConfig(modal=100000000)
        engine_name = "INTRADAY GORENGAN ENGINE"
    else:
        engine_class = InvestasiEngine
        config = InvestasiConfig(modal=100000000)
        engine_name = "INVESTASI ENGINE"

    print(f"\n🔧 Validating: {engine_name} dengan {len(data_dict)} saham (AGGRESSIVE 3%)")

    suite = ValidationSuite(engine_class, config, data_dict)
    results = suite.run_all()

    print("\n" + "="*80)
    print("✅ PHASE 2 VALIDATION COMPLETE - AGGRESSIVE")
    print("   File output: monte_carlo_optimal_aggressive.png")
    print("="*80)

# =============================================================================
# 30. MAIN MENU
# =============================================================================

def main():
    print("\n" + "="*70)
    print("🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE (AGGRESSIVE)")
    print("   FULL SYSTEM DENGAN DATA WAREHOUSE")
    print("   Modal-Adaptive Filters | Risk 3% per trade | Target 15-20% per tahun")
    print("   Market Regime Detection dengan Confidence Score")
    print("   Kelly Criterion | Correlation Check | Stress Testing")
    print("   Dividend Analyzer untuk Investasi Engine")
    print("   Export Google Sheets | Panduan Portfolio 3 Terbaik")
    print("="*70)

    sheets_exporter = GoogleSheetsExporter()
    sheets_exporter.ensure_spreadsheet_exists()

    print("\nPilih Mode:")
    print("1. Phase 1 - Trading Scanner (Live Signals) - AGGRESSIVE")
    print("2. Phase 2 - Validation Suite (Monte Carlo Optimal) - AGGRESSIVE")
    print("3. Inisialisasi Data Warehouse (Download historis lengkap)")
    print("4. Exit")

    while True:
        choice = input("\nPilihan (1/2/3/4): ").strip()
        if choice == '1':
            run_phase1(sheets_exporter)
            break
        elif choice == '2':
            run_phase2()
            break
        elif choice == '3':
            initialize_data_warehouse()
            break
        elif choice == '4':
            print("\n✅ Exiting...")
            break
        else:
            print("❌ Pilih 1, 2, 3, atau 4")

if __name__ == "__main__":
    main()

✅ Dependencies imported

🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE (AGGRESSIVE)
   FULL SYSTEM DENGAN DATA WAREHOUSE
   Modal-Adaptive Filters | Risk 3% per trade | Target 15-20% per tahun
   Market Regime Detection dengan Confidence Score
   Kelly Criterion | Correlation Check | Stress Testing
   Dividend Analyzer untuk Investasi Engine
   Export Google Sheets | Panduan Portfolio 3 Terbaik
✅ Berhasil konek ke Google Sheets
✅ Membuka spreadsheet: IDX_Scanner_All_Engines

📊 Google Sheets URL: https://docs.google.com/spreadsheets/d/10ITzuSx_6512KrfSmkqaRPaAsFbPi6Z_NO-6e6u9BKg

Pilih Mode:
1. Phase 1 - Trading Scanner (Live Signals) - AGGRESSIVE
2. Phase 2 - Validation Suite (Monte Carlo Optimal) - AGGRESSIVE
3. Inisialisasi Data Warehouse (Download historis lengkap)
4. Exit

Pilihan (1/2/3/4): 1

🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE (AGGRESSIVE)
   Modal-Adaptive Filters | Risk 3% per trade | Target 15-20% per tahun
   Export: Google Sheets | Panduan Portfolio 3 Terbaik

📊 Data warehouse 